In [1]:
# ============================================================
# DATATHON 2026 — NON-LOGIN EXTRA 3 TABLES
# Disk-safe / OOM-safe
#
# Mục tiêu tạo 3 bảng insight:
#
# TABLE 1:
#   Non-login entry -> login -> strict contact path
#   Xem first non-login event là category/city/surface gì,
#   rồi sau login contact vào category/city/event_type gì.
#
# TABLE 2:
#   Tết impact strict contact rate
#   So sánh pre_tet_7d / tet_7d / post_tet_7d,
#   không chỉ nhìn event rows.
#
# TABLE 3:
#   Same-session transfer
#   Item/category đã xem trước login có xuất hiện lại sau login/contact không.
#
# Outputs:
#   /kaggle/working/eda_non_login_extra_3_tables/tables
#   /kaggle/working/eda_non_login_extra_3_tables/cache
# ============================================================

import os
import glob
import gc
import sys
import shutil
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ============================================================
# 0. CONFIG
# ============================================================

BASE_PATH_2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"
EVENTS_PATH = os.path.join(BASE_PATH_2, "fact_user_events")

OUTPUT_DIR = "/kaggle/working/eda_non_login_extra_3_tables"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")

CLEAN_OUTPUT = True

if CLEAN_OUTPUT and os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

PARTIAL_SESSION_DIR = os.path.join(CACHE_DIR, "partial_session_sample")
PARTIAL_DAILY_DIR = os.path.join(CACHE_DIR, "partial_daily_full")
PARTIAL_TARGET_EVENT_DIR = os.path.join(CACHE_DIR, "partial_target_session_events")

for d in [PARTIAL_SESSION_DIR, PARTIAL_DAILY_DIR, PARTIAL_TARGET_EVENT_DIR]:
    os.makedirs(d, exist_ok=True)

EVENT_FILES = sorted(glob.glob(os.path.join(EVENTS_PATH, "*.parquet")))

BATCH_SIZE = 300_000

# Session-level phải sample để không nổ RAM/disk.
# 2 = giữ 2% session. Nếu vẫn nặng thì đổi thành 1.
# Nếu muốn chắc hơn và disk còn nhiều thì đổi thành 5.
SESSION_SAMPLE_MOD = 100
SESSION_SAMPLE_KEEP = 2

# Tết 2026: mùng 1 khoảng 2026-02-17.
# Có thể chỉnh nếu bạn muốn window khác.
TET_DATE = pd.Timestamp("2026-02-17")
PRE_TET_DAYS = 7
TET_DAYS = 7
POST_TET_DAYS = 7

STRICT_CONTACT_EVENTS = {
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
}

EVENT_COLS = [
    "is_login",
    "user_id",
    "session_id",
    "event_id",
    "item_id",
    "city_name",
    "category",
    "event_type",
    "event_ts",
    "surface",
    "device",
    "dwell_time_sec",
    "is_contact",
    "date",
]

print("EVENTS_PATH:", EVENTS_PATH)
print("Event files:", len(EVENT_FILES))
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Session sample:", f"{SESSION_SAMPLE_KEEP}/{SESSION_SAMPLE_MOD} = {SESSION_SAMPLE_KEEP / SESSION_SAMPLE_MOD:.0%}")

if len(EVENT_FILES) == 0:
    raise FileNotFoundError(f"No parquet files found in {EVENTS_PATH}")

# ============================================================
# 1. HELPERS
# ============================================================

def clean_text_series(s, default="unknown"):
    return (
        s.astype("string")
         .fillna(default)
         .str.strip()
         .str.lower()
         .replace("", default)
    )

def normalize_login_group(s):
    x = clean_text_series(s)
    return np.where(x == "login", "login", "non_login")

def category_name_from_series(s):
    s_num = pd.to_numeric(s, errors="coerce").astype("Int64")
    mp = {
        1010: "1010_room_rental",
        1020: "1020_apartment",
        1030: "1030_house",
        1040: "1040_land_commercial",
        1050: "1050_new_project",
    }
    out = s_num.map(mp)
    return out.fillna("unknown_" + s_num.astype(str).fillna("NULL"))

def session_sample_mask(session_series, mod=100, keep=2):
    h = pd.util.hash_pandas_object(session_series.astype("string"), index=False).values
    return (h % mod) < keep

def write_parquet(df, path):
    df.to_parquet(path, index=False, compression="zstd")

def save_df(df, name, show_head=30):
    csv_path = os.path.join(TABLE_DIR, f"{name}.csv")
    parquet_path = os.path.join(TABLE_DIR, f"{name}.parquet")

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    try:
        df.to_parquet(parquet_path, index=False, compression="zstd")
    except Exception as e:
        print("[WARN] Cannot save parquet:", name, e)

    print("\n[SAVE TABLE]", csv_path)
    display(df.head(show_head))
    return df

def run_duck(query, filename=None, show_head=30, memory_limit="6GB"):
    con = duckdb.connect()
    con.execute("PRAGMA threads=2")
    con.execute(f"PRAGMA memory_limit='{memory_limit}'")
    con.execute("PRAGMA preserve_insertion_order=false")
    df = con.execute(query).df()
    con.close()

    if filename:
        return save_df(df, filename, show_head=show_head)
    return df

def glob_sql(path):
    return os.path.join(path, "*.parquet").replace("\\", "/")

# ============================================================
# 2. FIRST PASS
#    - FULL daily strict contact summary
#    - SAMPLE session gate summary
# ============================================================

part_id = 0

for file_idx, fp in enumerate(EVENT_FILES):
    print(f"\n[FIRST PASS] file {file_idx + 1}/{len(EVENT_FILES)}: {os.path.basename(fp)}")

    pf = pq.ParquetFile(fp)

    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE, columns=EVENT_COLS)):
        df = batch.to_pandas()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # -----------------------------
        # Clean base fields
        # -----------------------------
        df["is_login_group"] = normalize_login_group(df["is_login"])
        df["event_type_clean"] = clean_text_series(df["event_type"])
        df["active_date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("D")
        df["event_ts_clean"] = pd.to_datetime(df["event_ts"], errors="coerce")

        df["is_pageview_int"] = (df["event_type_clean"] == "pageview").astype("int8")
        df["is_strict_contact_int"] = df["event_type_clean"].isin(STRICT_CONTACT_EVENTS).astype("int8")
        df["is_contact_int"] = pd.to_numeric(df["is_contact"], errors="coerce").fillna(0).astype("int8")

        # -----------------------------
        # A. FULL daily table for Tet impact
        # -----------------------------
        daily_base = df[df["active_date"].notna()].copy()

        if len(daily_base) > 0:
            g_daily = (
                daily_base.groupby(
                    ["active_date", "is_login_group"],
                    observed=True,
                    dropna=False,
                )
                .agg(
                    event_rows=("event_type_clean", "size"),
                    pageview_rows=("is_pageview_int", "sum"),
                    strict_contact_event_rows=("is_strict_contact_int", "sum"),
                    contact_flag_rows=("is_contact_int", "sum"),
                )
                .reset_index()
            )

            write_parquet(
                g_daily,
                os.path.join(PARTIAL_DAILY_DIR, f"part_{part_id:07d}.parquet")
            )

        # -----------------------------
        # B. SAMPLE session summary for non-login -> login sessions
        # -----------------------------
        sess_df = df[df["session_id"].notna() & df["event_ts_clean"].notna()].copy()

        if len(sess_df) > 0:
            sess_df["session_id"] = sess_df["session_id"].astype("string")

            sess_df = sess_df[
                session_sample_mask(
                    sess_df["session_id"],
                    mod=SESSION_SAMPLE_MOD,
                    keep=SESSION_SAMPLE_KEEP,
                )
            ].copy()

            if len(sess_df) > 0:
                sess_df["login_ts"] = sess_df["event_ts_clean"].where(
                    sess_df["is_login_group"] == "login",
                    pd.NaT
                )
                sess_df["nonlogin_ts"] = sess_df["event_ts_clean"].where(
                    sess_df["is_login_group"] == "non_login",
                    pd.NaT
                )

                sess_df["login_rows"] = (sess_df["is_login_group"] == "login").astype("int8")
                sess_df["nonlogin_rows"] = (sess_df["is_login_group"] == "non_login").astype("int8")

                g_sess = (
                    sess_df.groupby("session_id", observed=True)
                    .agg(
                        first_event_ts=("event_ts_clean", "min"),
                        last_event_ts=("event_ts_clean", "max"),
                        first_login_ts=("login_ts", "min"),
                        first_nonlogin_ts=("nonlogin_ts", "min"),
                        event_rows=("event_type_clean", "size"),
                        login_event_rows=("login_rows", "sum"),
                        nonlogin_event_rows=("nonlogin_rows", "sum"),
                        pageview_rows=("is_pageview_int", "sum"),
                        strict_contact_event_rows=("is_strict_contact_int", "sum"),
                        contact_flag_rows=("is_contact_int", "sum"),
                    )
                    .reset_index()
                )

                write_parquet(
                    g_sess,
                    os.path.join(PARTIAL_SESSION_DIR, f"part_{part_id:07d}.parquet")
                )

        part_id += 1

        del df
        gc.collect()

print("\n[DONE FIRST PASS]")
print("Partial daily files:", len(glob.glob(glob_sql(PARTIAL_DAILY_DIR))))
print("Partial session files:", len(glob.glob(glob_sql(PARTIAL_SESSION_DIR))))

# ============================================================
# 3. REDUCE TABLE 2 — TET STRICT CONTACT RATE
# ============================================================

DAILY_GLOB = glob_sql(PARTIAL_DAILY_DIR)

daily_full = run_duck(f"""
SELECT
    CAST(active_date AS DATE) AS active_date,
    is_login_group,

    SUM(event_rows) AS event_rows,
    SUM(pageview_rows) AS pageview_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows,

    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS strict_contact_per_event,
    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(pageview_rows), 0) AS strict_contact_per_pageview,
    SUM(contact_flag_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS contact_flag_per_event
FROM read_parquet('{DAILY_GLOB}')
GROUP BY active_date, is_login_group
ORDER BY active_date, is_login_group
""", "02A_daily_strict_contact_rate_full", show_head=20)

pre_start = TET_DATE - pd.Timedelta(days=PRE_TET_DAYS)
pre_end = TET_DATE - pd.Timedelta(days=1)

tet_start = TET_DATE
tet_end = TET_DATE + pd.Timedelta(days=TET_DAYS - 1)

post_start = TET_DATE + pd.Timedelta(days=TET_DAYS)
post_end = post_start + pd.Timedelta(days=POST_TET_DAYS - 1)

daily_full["active_date"] = pd.to_datetime(daily_full["active_date"])

def tet_window(d):
    if pre_start <= d <= pre_end:
        return "01_pre_tet_7d"
    if tet_start <= d <= tet_end:
        return "02_tet_7d"
    if post_start <= d <= post_end:
        return "03_post_tet_7d"
    return "other"

daily_full["tet_window"] = daily_full["active_date"].apply(tet_window)

tet_impact = (
    daily_full[daily_full["tet_window"] != "other"]
    .groupby(["tet_window", "is_login_group"], as_index=False)
    .agg(
        days=("active_date", "nunique"),
        event_rows=("event_rows", "sum"),
        pageview_rows=("pageview_rows", "sum"),
        strict_contact_event_rows=("strict_contact_event_rows", "sum"),
        contact_flag_rows=("contact_flag_rows", "sum"),
        avg_daily_event_rows=("event_rows", "mean"),
        avg_daily_pageview_rows=("pageview_rows", "mean"),
        avg_daily_strict_contact_rows=("strict_contact_event_rows", "mean"),
    )
)

tet_impact["strict_contact_per_event"] = (
    tet_impact["strict_contact_event_rows"] / tet_impact["event_rows"].replace(0, np.nan)
)
tet_impact["strict_contact_per_pageview"] = (
    tet_impact["strict_contact_event_rows"] / tet_impact["pageview_rows"].replace(0, np.nan)
)
tet_impact["contact_flag_per_event"] = (
    tet_impact["contact_flag_rows"] / tet_impact["event_rows"].replace(0, np.nan)
)

# So sánh với pre_tet_7d trong từng login group
pre_ref = (
    tet_impact[tet_impact["tet_window"] == "01_pre_tet_7d"]
    [["is_login_group", "strict_contact_per_event", "strict_contact_per_pageview", "avg_daily_event_rows"]]
    .rename(columns={
        "strict_contact_per_event": "pre_ref_strict_contact_per_event",
        "strict_contact_per_pageview": "pre_ref_strict_contact_per_pageview",
        "avg_daily_event_rows": "pre_ref_avg_daily_event_rows",
    })
)

tet_impact = tet_impact.merge(pre_ref, on="is_login_group", how="left")

tet_impact["strict_contact_per_event_delta_vs_pre"] = (
    tet_impact["strict_contact_per_event"] - tet_impact["pre_ref_strict_contact_per_event"]
)
tet_impact["strict_contact_per_pageview_delta_vs_pre"] = (
    tet_impact["strict_contact_per_pageview"] - tet_impact["pre_ref_strict_contact_per_pageview"]
)
tet_impact["avg_daily_event_rows_delta_pct_vs_pre"] = (
    tet_impact["avg_daily_event_rows"] / tet_impact["pre_ref_avg_daily_event_rows"].replace(0, np.nan) - 1
)

tet_impact = tet_impact.sort_values(["tet_window", "is_login_group"])
save_df(tet_impact, "02_tet_impact_strict_contact_window_summary", show_head=50)

print("\nTET WINDOWS:")
print("pre_tet_7d :", pre_start.date(), "->", pre_end.date())
print("tet_7d     :", tet_start.date(), "->", tet_end.date())
print("post_tet_7d:", post_start.date(), "->", post_end.date())

# ============================================================
# 4. REDUCE SESSION GATES
#    Tìm session sample có pattern non-login -> login
# ============================================================

SESSION_GLOB = glob_sql(PARTIAL_SESSION_DIR)
SESSION_GATE_PATH = os.path.join(CACHE_DIR, "session_gate_sample.parquet").replace("\\", "/")
TARGET_SESSION_PATH = os.path.join(CACHE_DIR, "target_nonlogin_then_login_sessions.parquet").replace("\\", "/")

con = duckdb.connect()
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA preserve_insertion_order=false")

con.execute(f"""
COPY (
    WITH sess AS (
        SELECT
            CAST(session_id AS VARCHAR) AS session_id,

            MIN(first_event_ts) AS first_event_ts,
            MAX(last_event_ts) AS last_event_ts,
            MIN(first_login_ts) AS first_login_ts,
            MIN(first_nonlogin_ts) AS first_nonlogin_ts,

            SUM(event_rows) AS event_rows,
            SUM(login_event_rows) AS login_event_rows,
            SUM(nonlogin_event_rows) AS nonlogin_event_rows,
            SUM(pageview_rows) AS pageview_rows,
            SUM(strict_contact_event_rows) AS strict_contact_event_rows,
            SUM(contact_flag_rows) AS contact_flag_rows
        FROM read_parquet('{SESSION_GLOB}')
        GROUP BY session_id
    ),

    scored AS (
        SELECT
            *,
            CASE
                WHEN login_event_rows > 0
                 AND nonlogin_event_rows > 0
                 AND first_nonlogin_ts <= first_login_ts
                    THEN 1
                ELSE 0
            END AS is_nonlogin_then_login_session,

            DATE_DIFF('second', first_event_ts, last_event_ts) AS session_duration_sec
        FROM sess
    )

    SELECT *
    FROM scored
) TO '{SESSION_GATE_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{SESSION_GATE_PATH}')
    WHERE is_nonlogin_then_login_session = 1
) TO '{TARGET_SESSION_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

session_gate_summary = con.execute(f"""
SELECT
    CASE
        WHEN login_event_rows > 0 AND nonlogin_event_rows = 0 THEN 'login_only'
        WHEN nonlogin_event_rows > 0 AND login_event_rows = 0 THEN 'non_login_only'
        WHEN is_nonlogin_then_login_session = 1 THEN 'nonlogin_then_login'
        WHEN login_event_rows > 0 AND nonlogin_event_rows > 0 THEN 'mixed_other_order'
        ELSE 'unknown'
    END AS session_type,

    COUNT(*) AS sampled_sessions,
    SUM(event_rows) AS event_rows,
    AVG(event_rows) AS avg_event_rows_per_session,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_strict_contact,
    SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS strict_contact_session_rate
FROM read_parquet('{SESSION_GATE_PATH}')
GROUP BY session_type
ORDER BY sampled_sessions DESC
""").df()

con.close()

save_df(session_gate_summary, "00_session_gate_summary_sample", show_head=50)

target_sessions = pd.read_parquet(TARGET_SESSION_PATH)
target_sessions["session_id"] = target_sessions["session_id"].astype("string")

print("\nTarget non-login -> login sampled sessions:", len(target_sessions))

if len(target_sessions) == 0:
    raise ValueError("Không có target session non-login -> login trong sample. Tăng SESSION_SAMPLE_KEEP hoặc check data.")

# Dictionary nhỏ để filter batch nhanh
target_session_set = set(target_sessions["session_id"].astype(str).tolist())
target_info = target_sessions[["session_id", "first_login_ts", "first_nonlogin_ts"]].copy()
target_info["session_id"] = target_info["session_id"].astype("string")
target_info["first_login_ts"] = pd.to_datetime(target_info["first_login_ts"], errors="coerce")
target_info["first_nonlogin_ts"] = pd.to_datetime(target_info["first_nonlogin_ts"], errors="coerce")

# ============================================================
# 5. SECOND PASS
#    Lấy event trong target sessions:
#    - pre_login non-login events
#    - post_login login events
# ============================================================

part_id = 0

for file_idx, fp in enumerate(EVENT_FILES):
    print(f"\n[SECOND PASS] file {file_idx + 1}/{len(EVENT_FILES)}: {os.path.basename(fp)}")

    pf = pq.ParquetFile(fp)

    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE, columns=EVENT_COLS)):
        df = batch.to_pandas()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        if "session_id" not in df.columns:
            del df
            gc.collect()
            continue

        df = df[df["session_id"].notna()].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["session_id"] = df["session_id"].astype("string")

        # Chỉ giữ target non-login -> login sessions
        df = df[df["session_id"].astype(str).isin(target_session_set)].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # Merge first_login_ts
        df = df.merge(target_info, on="session_id", how="inner")

        df["is_login_group"] = normalize_login_group(df["is_login"])
        df["event_type_clean"] = clean_text_series(df["event_type"])
        df["city_clean"] = clean_text_series(df["city_name"])
        df["surface_clean"] = clean_text_series(df["surface"])
        df["device_clean"] = clean_text_series(df["device"])

        df["category"] = pd.to_numeric(df["category"], errors="coerce").astype("Int64")
        df["category_name"] = category_name_from_series(df["category"])

        df["event_ts_clean"] = pd.to_datetime(df["event_ts"], errors="coerce")
        df = df[df["event_ts_clean"].notna() & df["first_login_ts"].notna()].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["is_pageview_int"] = (df["event_type_clean"] == "pageview").astype("int8")
        df["is_strict_contact_int"] = df["event_type_clean"].isin(STRICT_CONTACT_EVENTS).astype("int8")
        df["is_contact_int"] = pd.to_numeric(df["is_contact"], errors="coerce").fillna(0).astype("int8")

        # item_id clean để tránh null làm join lỗi
        df["item_id_clean"] = (
            df["item_id"]
            .astype("string")
            .fillna("unknown_item")
            .str.strip()
            .replace("", "unknown_item")
        )

        # Phase:
        # pre_nonlogin = hành vi non-login trước hoặc đúng lúc first login
        # post_login   = hành vi login sau hoặc đúng lúc first login
        pre_mask = (
            (df["is_login_group"] == "non_login")
            & (df["event_ts_clean"] <= df["first_login_ts"])
        )

        post_mask = (
            (df["is_login_group"] == "login")
            & (df["event_ts_clean"] >= df["first_login_ts"])
        )

        df = df[pre_mask | post_mask].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["phase"] = np.where(pre_mask.loc[df.index], "pre_nonlogin", "post_login")

        keep_cols = [
            "session_id",
            "phase",
            "event_ts_clean",
            "event_type_clean",
            "item_id_clean",
            "city_clean",
            "category",
            "category_name",
            "surface_clean",
            "device_clean",
            "is_pageview_int",
            "is_strict_contact_int",
            "is_contact_int",
            "first_login_ts",
            "first_nonlogin_ts",
        ]

        out = df[keep_cols].copy()

        write_parquet(
            out,
            os.path.join(PARTIAL_TARGET_EVENT_DIR, f"part_{part_id:07d}.parquet")
        )

        part_id += 1

        del df, out
        gc.collect()

print("\n[DONE SECOND PASS]")
print("Partial target event files:", len(glob.glob(glob_sql(PARTIAL_TARGET_EVENT_DIR))))

TARGET_EVENT_GLOB = glob_sql(PARTIAL_TARGET_EVENT_DIR)

if len(glob.glob(TARGET_EVENT_GLOB)) == 0:
    raise ValueError("Không có target event file. Có thể SESSION_SAMPLE_KEEP quá nhỏ hoặc filter quá chặt.")

# ============================================================
# 6. REDUCE TABLE 1 + TABLE 3
# ============================================================

SESSION_FLOW_DETAIL_PATH = os.path.join(CACHE_DIR, "session_flow_detail.parquet").replace("\\", "/")

con = duckdb.connect()
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='8GB'")
con.execute("PRAGMA preserve_insertion_order=false")

con.execute(f"""
COPY (
    WITH e AS (
        SELECT
            CAST(session_id AS VARCHAR) AS session_id,
            phase,
            CAST(event_ts_clean AS TIMESTAMP) AS event_ts,
            event_type_clean,
            item_id_clean,
            city_clean,
            CAST(category AS BIGINT) AS category,
            category_name,
            surface_clean,
            device_clean,
            is_pageview_int,
            is_strict_contact_int,
            is_contact_int,
            CAST(first_login_ts AS TIMESTAMP) AS first_login_ts,
            CAST(first_nonlogin_ts AS TIMESTAMP) AS first_nonlogin_ts
        FROM read_parquet('{TARGET_EVENT_GLOB}')
    ),

    pre AS (
        SELECT
            session_id,

            COUNT(*) AS pre_event_rows,
            SUM(is_pageview_int) AS pre_pageview_rows,
            COUNT(DISTINCT CASE WHEN item_id_clean <> 'unknown_item' THEN item_id_clean END) AS pre_distinct_items,
            COUNT(DISTINCT category_name) AS pre_distinct_categories,

            ARG_MIN(event_type_clean, event_ts) AS first_pre_event_type,
            ARG_MIN(item_id_clean, event_ts) AS first_pre_item_id,
            ARG_MIN(category_name, event_ts) AS first_pre_category_name,
            ARG_MIN(city_clean, event_ts) AS first_pre_city,
            ARG_MIN(surface_clean, event_ts) AS first_pre_surface,
            ARG_MIN(device_clean, event_ts) AS first_pre_device,
            MIN(event_ts) AS first_pre_event_ts
        FROM e
        WHERE phase = 'pre_nonlogin'
        GROUP BY session_id
    ),

    post AS (
        SELECT
            session_id,

            COUNT(*) AS post_event_rows,
            SUM(is_pageview_int) AS post_pageview_rows,
            SUM(is_strict_contact_int) AS post_strict_contact_rows,
            COUNT(DISTINCT CASE WHEN item_id_clean <> 'unknown_item' THEN item_id_clean END) AS post_distinct_items,
            COUNT(DISTINCT category_name) AS post_distinct_categories,

            ARG_MIN(event_type_clean, event_ts) AS first_post_event_type,
            ARG_MIN(item_id_clean, event_ts) AS first_post_item_id,
            ARG_MIN(category_name, event_ts) AS first_post_category_name,
            ARG_MIN(city_clean, event_ts) AS first_post_city,
            ARG_MIN(surface_clean, event_ts) AS first_post_surface,
            ARG_MIN(device_clean, event_ts) AS first_post_device,
            MIN(event_ts) AS first_post_event_ts
        FROM e
        WHERE phase = 'post_login'
        GROUP BY session_id
    ),

    post_contact AS (
        SELECT
            session_id,

            COUNT(*) AS post_contact_rows,
            COUNT(DISTINCT CASE WHEN item_id_clean <> 'unknown_item' THEN item_id_clean END) AS post_contact_distinct_items,
            COUNT(DISTINCT category_name) AS post_contact_distinct_categories,

            ARG_MIN(event_type_clean, event_ts) AS first_post_contact_event_type,
            ARG_MIN(item_id_clean, event_ts) AS first_post_contact_item_id,
            ARG_MIN(category_name, event_ts) AS first_post_contact_category_name,
            ARG_MIN(city_clean, event_ts) AS first_post_contact_city,
            ARG_MIN(surface_clean, event_ts) AS first_post_contact_surface,
            ARG_MIN(device_clean, event_ts) AS first_post_contact_device,
            MIN(event_ts) AS first_post_contact_ts
        FROM e
        WHERE phase = 'post_login'
          AND is_strict_contact_int = 1
        GROUP BY session_id
    ),

    pre_view_item AS (
        SELECT DISTINCT
            session_id,
            item_id_clean AS item_id,
            category_name
        FROM e
        WHERE phase = 'pre_nonlogin'
          AND is_pageview_int = 1
          AND item_id_clean <> 'unknown_item'
    ),

    pre_view_cat AS (
        SELECT DISTINCT
            session_id,
            category_name
        FROM e
        WHERE phase = 'pre_nonlogin'
          AND is_pageview_int = 1
          AND category_name IS NOT NULL
    ),

    post_any_item AS (
        SELECT DISTINCT
            session_id,
            item_id_clean AS item_id,
            category_name
        FROM e
        WHERE phase = 'post_login'
          AND item_id_clean <> 'unknown_item'
    ),

    post_view_item AS (
        SELECT DISTINCT
            session_id,
            item_id_clean AS item_id,
            category_name
        FROM e
        WHERE phase = 'post_login'
          AND is_pageview_int = 1
          AND item_id_clean <> 'unknown_item'
    ),

    post_contact_item AS (
        SELECT DISTINCT
            session_id,
            item_id_clean AS item_id,
            category_name
        FROM e
        WHERE phase = 'post_login'
          AND is_strict_contact_int = 1
          AND item_id_clean <> 'unknown_item'
    ),

    post_any_cat AS (
        SELECT DISTINCT
            session_id,
            category_name
        FROM e
        WHERE phase = 'post_login'
          AND category_name IS NOT NULL
    ),

    post_contact_cat AS (
        SELECT DISTINCT
            session_id,
            category_name
        FROM e
        WHERE phase = 'post_login'
          AND is_strict_contact_int = 1
          AND category_name IS NOT NULL
    ),

    item_transfer AS (
        SELECT
            p.session_id,

            COUNT(DISTINCT p.item_id) AS pre_view_items,

            COUNT(DISTINCT CASE WHEN pa.item_id IS NOT NULL THEN p.item_id END)
                AS pre_view_items_reappear_any_after_login,

            COUNT(DISTINCT CASE WHEN pv.item_id IS NOT NULL THEN p.item_id END)
                AS pre_view_items_reappear_as_pageview_after_login,

            COUNT(DISTINCT CASE WHEN pc.item_id IS NOT NULL THEN p.item_id END)
                AS pre_view_items_contacted_after_login
        FROM pre_view_item p
        LEFT JOIN post_any_item pa
            ON p.session_id = pa.session_id
           AND p.item_id = pa.item_id
        LEFT JOIN post_view_item pv
            ON p.session_id = pv.session_id
           AND p.item_id = pv.item_id
        LEFT JOIN post_contact_item pc
            ON p.session_id = pc.session_id
           AND p.item_id = pc.item_id
        GROUP BY p.session_id
    ),

    category_transfer AS (
        SELECT
            p.session_id,

            COUNT(DISTINCT p.category_name) AS pre_view_categories,

            COUNT(DISTINCT CASE WHEN pa.category_name IS NOT NULL THEN p.category_name END)
                AS pre_view_categories_reappear_any_after_login,

            COUNT(DISTINCT CASE WHEN pc.category_name IS NOT NULL THEN p.category_name END)
                AS pre_view_categories_contacted_after_login
        FROM pre_view_cat p
        LEFT JOIN post_any_cat pa
            ON p.session_id = pa.session_id
           AND p.category_name = pa.category_name
        LEFT JOIN post_contact_cat pc
            ON p.session_id = pc.session_id
           AND p.category_name = pc.category_name
        GROUP BY p.session_id
    )

    SELECT
        COALESCE(pre.session_id, post.session_id, post_contact.session_id) AS session_id,

        pre.first_pre_event_ts,
        post.first_post_event_ts,
        post_contact.first_post_contact_ts,

        pre.pre_event_rows,
        pre.pre_pageview_rows,
        pre.pre_distinct_items,
        pre.pre_distinct_categories,

        post.post_event_rows,
        post.post_pageview_rows,
        post.post_strict_contact_rows,
        post.post_distinct_items,
        post.post_distinct_categories,

        post_contact.post_contact_rows,
        post_contact.post_contact_distinct_items,
        post_contact.post_contact_distinct_categories,

        pre.first_pre_event_type,
        pre.first_pre_item_id,
        pre.first_pre_category_name,
        pre.first_pre_city,
        pre.first_pre_surface,
        pre.first_pre_device,

        post.first_post_event_type,
        post.first_post_item_id,
        post.first_post_category_name,
        post.first_post_city,
        post.first_post_surface,
        post.first_post_device,

        post_contact.first_post_contact_event_type,
        post_contact.first_post_contact_item_id,
        post_contact.first_post_contact_category_name,
        post_contact.first_post_contact_city,
        post_contact.first_post_contact_surface,
        post_contact.first_post_contact_device,

        COALESCE(item_transfer.pre_view_items, 0) AS pre_view_items,
        COALESCE(item_transfer.pre_view_items_reappear_any_after_login, 0)
            AS pre_view_items_reappear_any_after_login,
        COALESCE(item_transfer.pre_view_items_reappear_as_pageview_after_login, 0)
            AS pre_view_items_reappear_as_pageview_after_login,
        COALESCE(item_transfer.pre_view_items_contacted_after_login, 0)
            AS pre_view_items_contacted_after_login,

        COALESCE(category_transfer.pre_view_categories, 0) AS pre_view_categories,
        COALESCE(category_transfer.pre_view_categories_reappear_any_after_login, 0)
            AS pre_view_categories_reappear_any_after_login,
        COALESCE(category_transfer.pre_view_categories_contacted_after_login, 0)
            AS pre_view_categories_contacted_after_login,

        CASE WHEN post.post_event_rows > 0 THEN 1 ELSE 0 END AS has_post_login_event,
        CASE WHEN post_contact.post_contact_rows > 0 THEN 1 ELSE 0 END AS has_strict_contact_after_login,

        CASE
            WHEN post_contact.first_post_contact_category_name IS NOT NULL
             AND pre.first_pre_category_name = post_contact.first_post_contact_category_name
                THEN 1 ELSE 0
        END AS first_pre_category_same_as_first_contact_category,

        CASE
            WHEN post_contact.first_post_contact_city IS NOT NULL
             AND pre.first_pre_city = post_contact.first_post_contact_city
                THEN 1 ELSE 0
        END AS first_pre_city_same_as_first_contact_city,

        CASE
            WHEN COALESCE(item_transfer.pre_view_items_reappear_any_after_login, 0) > 0
                THEN 1 ELSE 0
        END AS has_same_item_after_login,

        CASE
            WHEN COALESCE(item_transfer.pre_view_items_contacted_after_login, 0) > 0
                THEN 1 ELSE 0
        END AS has_same_item_contact_after_login,

        CASE
            WHEN COALESCE(category_transfer.pre_view_categories_reappear_any_after_login, 0) > 0
                THEN 1 ELSE 0
        END AS has_same_category_after_login,

        CASE
            WHEN COALESCE(category_transfer.pre_view_categories_contacted_after_login, 0) > 0
                THEN 1 ELSE 0
        END AS has_same_category_contact_after_login

    FROM pre
    FULL OUTER JOIN post
        ON pre.session_id = post.session_id
    FULL OUTER JOIN post_contact
        ON COALESCE(pre.session_id, post.session_id) = post_contact.session_id
    LEFT JOIN item_transfer
        ON COALESCE(pre.session_id, post.session_id, post_contact.session_id) = item_transfer.session_id
    LEFT JOIN category_transfer
        ON COALESCE(pre.session_id, post.session_id, post_contact.session_id) = category_transfer.session_id
) TO '{SESSION_FLOW_DETAIL_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.close()

# Lưu detail để debug sâu hơn nếu cần
session_flow_detail = run_duck(f"""
SELECT *
FROM read_parquet('{SESSION_FLOW_DETAIL_PATH}')
ORDER BY first_pre_event_ts
""", "99_session_flow_detail_nonlogin_then_login_sample", show_head=30, memory_limit="8GB")

# ============================================================
# TABLE 1:
# Non-login entry -> login -> strict contact path
# ============================================================

table1 = run_duck(f"""
SELECT
    COALESCE(first_pre_category_name, 'unknown') AS first_nonlogin_category,
    COALESCE(first_pre_city, 'unknown') AS first_nonlogin_city,
    COALESCE(first_pre_surface, 'unknown') AS first_nonlogin_surface,
    COALESCE(first_pre_device, 'unknown') AS first_nonlogin_device,

    COALESCE(first_post_contact_event_type, 'no_strict_contact_after_login') AS first_contact_after_login_event_type,
    COALESCE(first_post_contact_category_name, 'no_strict_contact_after_login') AS first_contact_after_login_category,
    COALESCE(first_post_contact_city, 'no_strict_contact_after_login') AS first_contact_after_login_city,

    COUNT(*) AS sampled_sessions,

    SUM(has_strict_contact_after_login) AS sessions_with_strict_contact_after_login,
    SUM(has_strict_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS strict_contact_after_login_session_rate,

    AVG(pre_event_rows) AS avg_pre_nonlogin_events,
    AVG(pre_pageview_rows) AS avg_pre_nonlogin_pageviews,
    AVG(post_event_rows) AS avg_post_login_events,
    AVG(post_pageview_rows) AS avg_post_login_pageviews,
    AVG(post_contact_rows) AS avg_post_login_strict_contacts,

    SUM(first_pre_category_same_as_first_contact_category) AS sessions_first_category_same_as_contact_category,
    SUM(first_pre_category_same_as_first_contact_category) * 1.0 / NULLIF(SUM(has_strict_contact_after_login), 0)
        AS same_first_category_among_contact_sessions_rate,

    SUM(first_pre_city_same_as_first_contact_city) AS sessions_first_city_same_as_contact_city,
    SUM(first_pre_city_same_as_first_contact_city) * 1.0 / NULLIF(SUM(has_strict_contact_after_login), 0)
        AS same_first_city_among_contact_sessions_rate,

    SUM(has_same_item_contact_after_login) AS sessions_contacted_same_item_seen_prelogin,
    SUM(has_same_item_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS contacted_same_item_seen_prelogin_session_rate,

    SUM(has_same_category_contact_after_login) AS sessions_contacted_same_category_seen_prelogin,
    SUM(has_same_category_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS contacted_same_category_seen_prelogin_session_rate

FROM read_parquet('{SESSION_FLOW_DETAIL_PATH}')
GROUP BY
    first_nonlogin_category,
    first_nonlogin_city,
    first_nonlogin_surface,
    first_nonlogin_device,
    first_contact_after_login_event_type,
    first_contact_after_login_category,
    first_contact_after_login_city
ORDER BY sampled_sessions DESC
LIMIT 300
""", "01_nonlogin_entry_to_login_contact_path_summary", show_head=80, memory_limit="8GB")

# ============================================================
# TABLE 3:
# Same-session transfer summary
# ============================================================

table3 = run_duck(f"""
SELECT
    COALESCE(first_pre_category_name, 'unknown') AS first_nonlogin_category,
    COALESCE(first_pre_city, 'unknown') AS first_nonlogin_city,
    COALESCE(first_pre_surface, 'unknown') AS first_nonlogin_surface,
    COALESCE(first_pre_device, 'unknown') AS first_nonlogin_device,

    COUNT(*) AS sampled_sessions,

    SUM(has_post_login_event) AS sessions_with_post_login_event,
    SUM(has_strict_contact_after_login) AS sessions_with_strict_contact_after_login,

    SUM(has_strict_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS strict_contact_after_login_session_rate,

    AVG(pre_view_items) AS avg_prelogin_view_items,
    AVG(pre_view_categories) AS avg_prelogin_view_categories,

    SUM(has_same_item_after_login) AS sessions_same_item_reappears_after_login,
    SUM(has_same_item_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_item_reappears_after_login_rate,

    SUM(has_same_item_contact_after_login) AS sessions_same_item_contact_after_login,
    SUM(has_same_item_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_item_contact_after_login_rate,

    SUM(has_same_category_after_login) AS sessions_same_category_reappears_after_login,
    SUM(has_same_category_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_category_reappears_after_login_rate,

    SUM(has_same_category_contact_after_login) AS sessions_same_category_contact_after_login,
    SUM(has_same_category_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_category_contact_after_login_rate,

    AVG(pre_view_items_reappear_any_after_login) AS avg_pre_items_reappear_after_login,
    AVG(pre_view_items_contacted_after_login) AS avg_pre_items_contacted_after_login,
    AVG(pre_view_categories_reappear_any_after_login) AS avg_pre_categories_reappear_after_login,
    AVG(pre_view_categories_contacted_after_login) AS avg_pre_categories_contacted_after_login

FROM read_parquet('{SESSION_FLOW_DETAIL_PATH}')
GROUP BY
    first_nonlogin_category,
    first_nonlogin_city,
    first_nonlogin_surface,
    first_nonlogin_device
ORDER BY sampled_sessions DESC
LIMIT 300
""", "03_same_session_prelogin_transfer_summary", show_head=80, memory_limit="8GB")

# ============================================================
# 7. QUICK OVERALL SUMMARY
# ============================================================

overall = run_duck(f"""
SELECT
    COUNT(*) AS target_nonlogin_then_login_sampled_sessions,

    SUM(has_strict_contact_after_login) AS sessions_with_strict_contact_after_login,
    SUM(has_strict_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS strict_contact_after_login_session_rate,

    SUM(has_same_item_after_login) AS sessions_same_item_reappears_after_login,
    SUM(has_same_item_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_item_reappears_after_login_rate,

    SUM(has_same_item_contact_after_login) AS sessions_same_item_contact_after_login,
    SUM(has_same_item_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_item_contact_after_login_rate,

    SUM(has_same_category_after_login) AS sessions_same_category_reappears_after_login,
    SUM(has_same_category_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_category_reappears_after_login_rate,

    SUM(has_same_category_contact_after_login) AS sessions_same_category_contact_after_login,
    SUM(has_same_category_contact_after_login) * 1.0 / NULLIF(COUNT(*), 0)
        AS same_category_contact_after_login_rate,

    AVG(pre_event_rows) AS avg_pre_nonlogin_events,
    AVG(pre_pageview_rows) AS avg_pre_nonlogin_pageviews,
    AVG(post_event_rows) AS avg_post_login_events,
    AVG(post_pageview_rows) AS avg_post_login_pageviews,
    AVG(post_contact_rows) AS avg_post_login_strict_contacts

FROM read_parquet('{SESSION_FLOW_DETAIL_PATH}')
""", "04_overall_nonlogin_to_login_transfer_summary", show_head=20, memory_limit="8GB")

print("\nDONE.")
print("Output folder:", OUTPUT_DIR)
print("Tables folder:", TABLE_DIR)

print("""
MAIN OUTPUT TABLES:

1) 01_nonlogin_entry_to_login_contact_path_summary.csv
   - First non-login category/city/surface/device
   - First strict contact after login
   - Contact rate and same category/city/item indicators

2) 02_tet_impact_strict_contact_window_summary.csv
   - pre_tet_7d / tet_7d / post_tet_7d
   - strict_contact_per_event
   - strict_contact_per_pageview
   - delta vs pre-tet

3) 03_same_session_prelogin_transfer_summary.csv
   - Whether pre-login viewed item/category reappears after login
   - Whether pre-login viewed item/category becomes strict contact after login

Debug/detail:
00_session_gate_summary_sample.csv
02A_daily_strict_contact_rate_full.csv
04_overall_nonlogin_to_login_transfer_summary.csv
99_session_flow_detail_nonlogin_then_login_sample.csv
""")

EVENTS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events
Event files: 500
OUTPUT_DIR: /kaggle/working/eda_non_login_extra_3_tables
Session sample: 2/100 = 2%

[FIRST PASS] file 1/500: datathon_fact_user_events-000000000000.parquet

[FIRST PASS] file 2/500: datathon_fact_user_events-000000000001.parquet

[FIRST PASS] file 3/500: datathon_fact_user_events-000000000002.parquet

[FIRST PASS] file 4/500: datathon_fact_user_events-000000000003.parquet

[FIRST PASS] file 5/500: datathon_fact_user_events-000000000004.parquet

[FIRST PASS] file 6/500: datathon_fact_user_events-000000000005.parquet

[FIRST PASS] file 7/500: datathon_fact_user_events-000000000006.parquet

[FIRST PASS] file 8/500: datathon_fact_user_events-000000000007.parquet

[FIRST PASS] file 9/500: datathon_fact_user_events-000000000008.parquet

[FIRST PASS] file 10/500: datathon_fact_user_events-000000000009.parquet

[FIRST PASS] file 11/500: datathon_fact_user_events-000000000010.parquet

[FIRST 

,active_date,is_login_group,event_rows,pageview_rows,strict_contact_event_rows,contact_flag_rows,strict_contact_per_event,strict_contact_per_pageview,contact_flag_per_event
0,2025-11-09,login,721524.0,397652.0,30784.0,323872.0,0.042665,0.077414,0.448872
1,2025-11-09,non_login,362788.0,110291.0,5150.0,252497.0,0.014196,0.046695,0.695990
2,2025-11-10,login,868961.0,422190.0,38199.0,446771.0,0.043959,0.090478,0.514144
3,2025-11-10,non_login,434770.0,124366.0,6437.0,310404.0,0.014806,0.051759,0.713950
4,2025-11-11,login,863448.0,411096.0,37345.0,452352.0,0.043251,0.090843,0.523890
5,2025-11-11,non_login,412610.0,113904.0,6201.0,298706.0,0.015029,0.054441,0.723943
6,2025-11-12,login,913734.0,409906.0,36589.0,503828.0,0.040043,0.089262,0.551395
7,2025-11-12,non_login,420828.0,115816.0,6082.0,305012.0,0.014452,0.052514,0.724790
8,2025-11-13,login,944823.0,410648.0,36206.0,534175.0,0.038320,0.088168,0.565370
9,2025-11-13,non_login,413327.0,113358.0,5884.0,299969.0,0.014236,0.051906,0.725743



[SAVE TABLE] /kaggle/working/eda_non_login_extra_3_tables/tables/02_tet_impact_strict_contact_window_summary.csv


,tet_window,is_login_group,days,event_rows,pageview_rows,strict_contact_event_rows,contact_flag_rows,avg_daily_event_rows,avg_daily_pageview_rows,avg_daily_strict_contact_rows,strict_contact_per_event,strict_contact_per_pageview,contact_flag_per_event,pre_ref_strict_contact_per_event,pre_ref_strict_contact_per_pageview,pre_ref_avg_daily_event_rows,strict_contact_per_event_delta_vs_pre,strict_contact_per_pageview_delta_vs_pre,avg_daily_event_rows_delta_pct_vs_pre
0,01_pre_tet_7d,login,7,1751257.0,914677.0,58058.0,836580.0,250179.571429,130668.142857,8294.000000,0.033152,0.063474,0.477703,0.033152,0.063474,250179.571429,0.000000,0.000000,0.000000
1,01_pre_tet_7d,non_login,7,1087933.0,286087.0,22750.0,801846.0,155419.000000,40869.571429,3250.000000,0.020911,0.079521,0.737036,0.020911,0.079521,155419.000000,0.000000,0.000000,0.000000
2,02_tet_7d,login,7,2919132.0,1575026.0,88469.0,1344106.0,417018.857143,225003.714286,12638.428571,0.030307,0.056170,0.460447,0.033152,0.063474,250179.571429,-0.002846,-0.007304,0.666878
3,02_tet_7d,non_login,7,1684354.0,445266.0,37216.0,1239088.0,240622.000000,63609.428571,5316.571429,0.022095,0.083581,0.735646,0.020911,0.079521,155419.000000,0.001184,0.004060,0.548215
4,03_post_tet_7d,login,7,6076236.0,3059008.0,234635.0,3017228.0,868033.714286,437001.142857,33519.285714,0.038615,0.076703,0.496562,0.033152,0.063474,250179.571429,0.005463,0.013229,2.469643
5,03_post_tet_7d,non_login,7,3586088.0,879638.0,88345.0,2706450.0,512298.285714,125662.571429,12620.714286,0.024635,0.100433,0.754708,0.020911,0.079521,155419.000000,0.003724,0.020912,2.296240



TET WINDOWS:
pre_tet_7d : 2026-02-10 -> 2026-02-16
tet_7d     : 2026-02-17 -> 2026-02-23
post_tet_7d: 2026-02-24 -> 2026-03-02


...


[SAVE TABLE] /kaggle/working/eda_non_login_extra_3_tables/tables/03_same_session_prelogin_transfer_summary.csv


,first_nonlogin_category,first_nonlogin_city,first_nonlogin_surface,first_nonlogin_device,sampled_sessions,sessions_with_post_login_event,sessions_with_strict_contact_after_login,strict_contact_after_login_session_rate,avg_prelogin_view_items,avg_prelogin_view_categories,...,sessions_same_item_contact_after_login,same_item_contact_after_login_rate,sessions_same_category_reappears_after_login,same_category_reappears_after_login_rate,sessions_same_category_contact_after_login,same_category_contact_after_login_rate,avg_pre_items_reappear_after_login,avg_pre_items_contacted_after_login,avg_pre_categories_reappear_after_login,avg_pre_categories_contacted_after_login
0,1020_apartment,tp hồ chí minh,ad_view,msite,3211,3211.0,715.0,0.222672,1.685145,1.089069,...,1.0,0.000311,1927.0,0.600125,346.0,0.107755,0.000934,0.000311,0.614762,0.108689
1,1050_new_project,tp hồ chí minh,ad_view,msite,2345,2345.0,548.0,0.233689,1.686994,1.113006,...,1.0,0.000426,852.0,0.363326,174.0,0.074200,0.003412,0.000426,0.373561,0.074627
2,1010_room_rental,tp hồ chí minh,ad_view,msite,1674,1674.0,377.0,0.225209,1.593190,1.123656,...,1.0,0.000597,487.0,0.290920,97.0,0.057945,0.008363,0.002389,0.302270,0.059737
3,1020_apartment,tp hồ chí minh,ad_view,desktop,1585,1585.0,389.0,0.245426,2.097792,1.174132,...,0.0,0.000000,995.0,0.627760,192.0,0.121136,0.001893,0.000000,0.664984,0.122397
4,1050_new_project,tp hồ chí minh,ad_view,desktop,1019,1019.0,230.0,0.225711,2.188420,1.218842,...,0.0,0.000000,400.0,0.392542,71.0,0.069676,0.000981,0.000000,0.412169,0.070658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,1050_new_project,bình dương,ad_view,desktop,21,21.0,9.0,0.428571,1.380952,1.142857,...,0.0,0.000000,8.0,0.380952,3.0,0.142857,0.000000,0.000000,0.380952,0.142857
76,1020_apartment,tiền giang,ad_view,msite,20,20.0,2.0,0.100000,1.700000,1.150000,...,0.0,0.000000,12.0,0.600000,0.0,0.000000,0.000000,0.000000,0.650000,0.000000
77,1010_room_rental,đồng nai,ad_view,msite,20,20.0,4.0,0.200000,1.450000,1.150000,...,0.0,0.000000,7.0,0.350000,2.0,0.100000,0.000000,0.000000,0.350000,0.100000
78,1020_apartment,hải phòng,ad_view,msite,19,19.0,6.0,0.315789,1.105263,1.052632,...,0.0,0.000000,9.0,0.473684,4.0,0.210526,0.000000,0.000000,0.473684,0.210526



[SAVE TABLE] /kaggle/working/eda_non_login_extra_3_tables/tables/04_overall_nonlogin_to_login_transfer_summary.csv


,target_nonlogin_then_login_sampled_sessions,sessions_with_strict_contact_after_login,strict_contact_after_login_session_rate,sessions_same_item_reappears_after_login,same_item_reappears_after_login_rate,sessions_same_item_contact_after_login,same_item_contact_after_login_rate,sessions_same_category_reappears_after_login,same_category_reappears_after_login_rate,sessions_same_category_contact_after_login,same_category_contact_after_login_rate,avg_pre_nonlogin_events,avg_pre_nonlogin_pageviews,avg_post_login_events,avg_post_login_pageviews,avg_post_login_strict_contacts
0,19253,4334.0,0.225108,28.0,0.001454,6.0,0.000312,8418.0,0.437231,1519.0,0.078897,6.214148,1.955591,19.587233,9.494988,3.347024



DONE.
Output folder: /kaggle/working/eda_non_login_extra_3_tables
Tables folder: /kaggle/working/eda_non_login_extra_3_tables/tables

MAIN OUTPUT TABLES:

1) 01_nonlogin_entry_to_login_contact_path_summary.csv
   - First non-login category/city/surface/device
   - First strict contact after login
   - Contact rate and same category/city/item indicators

2) 02_tet_impact_strict_contact_window_summary.csv
   - pre_tet_7d / tet_7d / post_tet_7d
   - strict_contact_per_event
   - strict_contact_per_pageview
   - delta vs pre-tet

3) 03_same_session_prelogin_transfer_summary.csv
   - Whether pre-login viewed item/category reappears after login
   - Whether pre-login viewed item/category becomes strict contact after login

Debug/detail:
00_session_gate_summary_sample.csv
02A_daily_strict_contact_rate_full.csv
04_overall_nonlogin_to_login_transfer_summary.csv
99_session_flow_detail_nonlogin_then_login_sample.csv



In [5]:
# ============================================================
# VISUALIZE NON-LOGIN EXTRA 3 TABLES
# Run AFTER:
# /kaggle/working/eda_non_login_extra_3_tables/tables
#
# Input tables:
# 00_session_gate_summary_sample
# 01_nonlogin_entry_to_login_contact_path_summary
# 02_tet_impact_strict_contact_window_summary
# 02A_daily_strict_contact_rate_full
# 03_same_session_prelogin_transfer_summary
# 04_overall_nonlogin_to_login_transfer_summary
#
# Output:
# /kaggle/working/eda_non_login_extra_3_tables_visuals/figures
# ============================================================

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ============================================================
# 0. CONFIG
# ============================================================

TABLE_DIR_CANDIDATES = [
    "/kaggle/working/eda_non_login_extra_3_tables/tables",
    "/kaggle/input",
]

OUTPUT_DIR = "/kaggle/working/eda_non_login_extra_3_tables_visuals"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLE_OUT_DIR = os.path.join(OUTPUT_DIR, "derived_tables")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_OUT_DIR, exist_ok=True)

TOP_N = 25
MIN_SESSIONS_FOR_RATE = 20

print("OUTPUT_DIR:", OUTPUT_DIR)

# ============================================================
# 1. HELPERS
# ============================================================

def find_table(stem):
    paths = []

    for root in TABLE_DIR_CANDIDATES:
        if not os.path.exists(root):
            continue

        paths += glob.glob(os.path.join(root, "**", f"{stem}.parquet"), recursive=True)
        paths += glob.glob(os.path.join(root, "**", f"{stem}.csv"), recursive=True)

    paths = sorted(set(paths))

    if len(paths) == 0:
        print("[MISSING]", stem)
        return None

    # prefer parquet
    paths = sorted(paths, key=lambda p: 0 if p.endswith(".parquet") else 1)
    return paths[0]

def read_table(stem):
    path = find_table(stem)

    if path is None:
        return None

    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)

    print("[LOAD]", stem, "| rows:", len(df), "|", path)
    return df

def save_derived(df, name):
    csv_path = os.path.join(TABLE_OUT_DIR, f"{name}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print("[SAVE DERIVED]", csv_path)
    display(df.head(30))
    return df

def plotly_safe_df(df):
    return df.copy().astype(object).where(pd.notna(df), None)

chart_index = []

def save_fig(fig, name, title=None):
    html_path = os.path.join(FIG_DIR, f"{name}.html")
    fig.write_html(html_path)
    print("[SAVE FIG]", html_path)

    chart_index.append({
        "name": name,
        "title": title if title else name,
        "html_path": html_path,
    })

    fig.show()

def to_num(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def add_pct_cols(df, rate_cols):
    df = df.copy()
    for c in rate_cols:
        if c in df.columns:
            df[c + "_pct"] = pd.to_numeric(df[c], errors="coerce") * 100
    return df

def clean_label(s):
    return (
        s.astype("string")
         .fillna("unknown")
         .str.replace("_", " ", regex=False)
    )

def weighted_rate(num, den):
    den = den.replace(0, np.nan)
    return num / den

def make_short_label(df, cols, sep=" | "):
    tmp = df.copy()
    parts = []
    for c in cols:
        if c in tmp.columns:
            parts.append(tmp[c].astype(str))
    if len(parts) == 0:
        return pd.Series(np.arange(len(tmp)).astype(str))
    out = parts[0]
    for p in parts[1:]:
        out = out + sep + p
    return out

# ============================================================
# 2. LOAD TABLES
# ============================================================

gate = read_table("00_session_gate_summary_sample")
path_tbl = read_table("01_nonlogin_entry_to_login_contact_path_summary")
tet = read_table("02_tet_impact_strict_contact_window_summary")
daily = read_table("02A_daily_strict_contact_rate_full")
transfer = read_table("03_same_session_prelogin_transfer_summary")
overall = read_table("04_overall_nonlogin_to_login_transfer_summary")

loaded_any = any(x is not None for x in [gate, path_tbl, tet, daily, transfer, overall])

if not loaded_any:
    print("\nKhông tìm thấy table nào. Kiểm tra output cũ:")
    files = []
    for root in ["/kaggle/working", "/kaggle/input"]:
        files += glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
        files += glob.glob(os.path.join(root, "**", "*.parquet"), recursive=True)

    manifest = pd.DataFrame({
        "path": files,
        "filename": [os.path.basename(p) for p in files],
        "folder": [os.path.dirname(p) for p in files],
        "size_mb": [os.path.getsize(p) / 1024 / 1024 for p in files],
    })
    display(manifest.sort_values("path").head(300))
    raise FileNotFoundError("Không tìm thấy output tables. Hãy chạy lại code tạo 3 bảng trước.")

# ============================================================
# 3. OVERALL / GATE SUMMARY
# ============================================================

if overall is not None and len(overall) > 0:
    df = overall.copy()
    df = to_num(df, df.columns)

    rate_cols = [
        "strict_contact_after_login_session_rate",
        "same_item_reappears_after_login_rate",
        "same_item_contact_after_login_rate",
        "same_category_reappears_after_login_rate",
        "same_category_contact_after_login_rate",
    ]
    df = add_pct_cols(df, rate_cols)

    # KPI cards
    row = df.iloc[0].to_dict()

    kpis = [
        ("Target sessions", row.get("target_nonlogin_then_login_sampled_sessions", np.nan)),
        ("Strict contact after login", row.get("strict_contact_after_login_session_rate_pct", np.nan)),
        ("Same item reappears", row.get("same_item_reappears_after_login_rate", np.nan) * 100),
        ("Same item contacted", row.get("same_item_contact_after_login_rate", np.nan) * 100),
        ("Same category contacted", row.get("same_category_contact_after_login_rate", np.nan) * 100),
        ("Avg pre-login events", row.get("avg_pre_nonlogin_events", np.nan)),
    ]

    fig = go.Figure()

    for i, (title, value) in enumerate(kpis):
        if "rate" in title.lower() or "contact" in title.lower() or "reappears" in title.lower():
            number_suffix = "%"
            number_value = value
            number_format = ".2f"
        else:
            number_suffix = ""
            number_value = value
            number_format = ",.0f" if title == "Target sessions" else ".2f"

        fig.add_trace(go.Indicator(
            mode="number",
            value=number_value if pd.notna(number_value) else 0,
            number={"suffix": number_suffix, "valueformat": number_format},
            title={"text": title},
            domain={
                "row": i // 3,
                "column": i % 3
            }
        ))

    fig.update_layout(
        grid={"rows": 2, "columns": 3, "pattern": "independent"},
        title="Overall: non-login → login transfer summary",
        height=520,
    )
    save_fig(fig, "00_overall_kpi_cards")

    # table view
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df.columns), align="left"),
        cells=dict(values=[df[c] for c in df.columns], align="left"),
    )])
    fig.update_layout(title="Overall transfer summary table", height=480)
    save_fig(fig, "00b_overall_summary_table")

if gate is not None and len(gate) > 0:
    df = gate.copy()
    df = to_num(df, [
        "sampled_sessions",
        "event_rows",
        "avg_event_rows_per_session",
        "strict_contact_event_rows",
        "sessions_with_strict_contact",
        "strict_contact_session_rate",
    ])
    df = add_pct_cols(df, ["strict_contact_session_rate"])

    fig = px.bar(
        plotly_safe_df(df.sort_values("sampled_sessions", ascending=False)),
        x="session_type",
        y="sampled_sessions",
        text="sampled_sessions",
        hover_data=[
            "event_rows",
            "avg_event_rows_per_session",
            "strict_contact_event_rows",
            "sessions_with_strict_contact",
            "strict_contact_session_rate_pct",
        ],
        title="Sampled sessions by session type",
    )
    fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
    fig.update_layout(
        xaxis_title="Session type",
        yaxis_title="Sampled sessions",
        xaxis_tickangle=-20,
        height=600,
    )
    save_fig(fig, "01_session_gate_volume")

    fig = px.bar(
        plotly_safe_df(df.sort_values("strict_contact_session_rate_pct", ascending=False)),
        x="session_type",
        y="strict_contact_session_rate_pct",
        text="strict_contact_session_rate_pct",
        hover_data=[
            "sampled_sessions",
            "strict_contact_event_rows",
            "sessions_with_strict_contact",
            "avg_event_rows_per_session",
        ],
        title="Strict-contact session rate by session type",
    )
    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        xaxis_title="Session type",
        yaxis_title="Sessions with strict contact (%)",
        xaxis_tickangle=-20,
        height=600,
    )
    save_fig(fig, "02_session_gate_strict_contact_rate")

# ============================================================
# 4. TABLE 2 — TET IMPACT VISUALIZATION
# ============================================================

if tet is not None and len(tet) > 0:
    df = tet.copy()
    df = to_num(df, [
        "days",
        "event_rows",
        "pageview_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "avg_daily_event_rows",
        "avg_daily_pageview_rows",
        "avg_daily_strict_contact_rows",
        "strict_contact_per_event",
        "strict_contact_per_pageview",
        "contact_flag_per_event",
        "strict_contact_per_event_delta_vs_pre",
        "strict_contact_per_pageview_delta_vs_pre",
        "avg_daily_event_rows_delta_pct_vs_pre",
    ])

    rate_cols = [
        "strict_contact_per_event",
        "strict_contact_per_pageview",
        "contact_flag_per_event",
        "strict_contact_per_event_delta_vs_pre",
        "strict_contact_per_pageview_delta_vs_pre",
        "avg_daily_event_rows_delta_pct_vs_pre",
    ]
    df = add_pct_cols(df, rate_cols)

    # 4.1 avg daily traffic by Tet window
    fig = px.bar(
        plotly_safe_df(df),
        x="tet_window",
        y="avg_daily_event_rows",
        color="is_login_group",
        barmode="group",
        hover_data=[
            "days",
            "event_rows",
            "pageview_rows",
            "avg_daily_pageview_rows",
            "avg_daily_event_rows_delta_pct_vs_pre_pct",
        ],
        title="Tết impact: average daily event rows",
    )
    fig.update_layout(
        xaxis_title="Window",
        yaxis_title="Avg daily event rows",
        height=600,
    )
    save_fig(fig, "03_tet_avg_daily_event_rows")

    # 4.2 strict contact per event
    fig = px.bar(
        plotly_safe_df(df),
        x="tet_window",
        y="strict_contact_per_event_pct",
        color="is_login_group",
        barmode="group",
        hover_data=[
            "event_rows",
            "strict_contact_event_rows",
            "strict_contact_per_event_delta_vs_pre_pct",
        ],
        title="Tết impact: strict contact per event",
    )
    fig.update_layout(
        xaxis_title="Window",
        yaxis_title="Strict contact / event (%)",
        height=600,
    )
    save_fig(fig, "04_tet_strict_contact_per_event")

    # 4.3 strict contact per pageview
    fig = px.bar(
        plotly_safe_df(df),
        x="tet_window",
        y="strict_contact_per_pageview_pct",
        color="is_login_group",
        barmode="group",
        hover_data=[
            "pageview_rows",
            "strict_contact_event_rows",
            "strict_contact_per_pageview_delta_vs_pre_pct",
        ],
        title="Tết impact: strict contact per pageview",
    )
    fig.update_layout(
        xaxis_title="Window",
        yaxis_title="Strict contact / pageview (%)",
        height=600,
    )
    save_fig(fig, "05_tet_strict_contact_per_pageview")

    # 4.4 delta vs pre tet
    plot_df = df[df["tet_window"] != "01_pre_tet_7d"].copy()
    fig = px.bar(
        plotly_safe_df(plot_df),
        x="tet_window",
        y=[
            "strict_contact_per_event_delta_vs_pre_pct",
            "strict_contact_per_pageview_delta_vs_pre_pct",
            "avg_daily_event_rows_delta_pct_vs_pre_pct",
        ],
        color="is_login_group",
        barmode="group",
        hover_data=[
            "event_rows",
            "pageview_rows",
            "strict_contact_event_rows",
        ],
        title="Tết impact: delta vs pre-Tết baseline",
    )
    fig.update_layout(
        xaxis_title="Window",
        yaxis_title="Delta vs pre-Tết, percentage points / %",
        height=650,
    )
    save_fig(fig, "06_tet_delta_vs_pre")

if daily is not None and len(daily) > 0:
    df = daily.copy()
    df["active_date"] = pd.to_datetime(df["active_date"], errors="coerce")
    df = df.sort_values(["active_date", "is_login_group"])
    df = to_num(df, [
        "event_rows",
        "pageview_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "strict_contact_per_event",
        "strict_contact_per_pageview",
        "contact_flag_per_event",
    ])

    df["strict_contact_per_event_pct"] = df["strict_contact_per_event"] * 100
    df["strict_contact_per_pageview_pct"] = df["strict_contact_per_pageview"] * 100

    # 4.5 daily event rows
    fig = px.line(
        plotly_safe_df(df),
        x="active_date",
        y="event_rows",
        color="is_login_group",
        hover_data=[
            "pageview_rows",
            "strict_contact_event_rows",
            "strict_contact_per_event_pct",
            "strict_contact_per_pageview_pct",
        ],
        title="Daily event rows: login vs non-login",
    )
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title="Event rows",
        height=620,
    )
    save_fig(fig, "07_daily_event_rows_login_nonlogin")

    # 4.6 daily contact rate
    fig = px.line(
        plotly_safe_df(df),
        x="active_date",
        y="strict_contact_per_pageview_pct",
        color="is_login_group",
        hover_data=[
            "event_rows",
            "pageview_rows",
            "strict_contact_event_rows",
            "strict_contact_per_event_pct",
        ],
        title="Daily strict contact per pageview",
    )
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title="Strict contact / pageview (%)",
        height=620,
    )
    save_fig(fig, "08_daily_strict_contact_per_pageview")

# ============================================================
# 5. TABLE 1 — PATH: ENTRY NON-LOGIN -> CONTACT AFTER LOGIN
# ============================================================

if path_tbl is not None and len(path_tbl) > 0:
    df = path_tbl.copy()

    df = to_num(df, [
        "sampled_sessions",
        "sessions_with_strict_contact_after_login",
        "strict_contact_after_login_session_rate",
        "avg_pre_nonlogin_events",
        "avg_pre_nonlogin_pageviews",
        "avg_post_login_events",
        "avg_post_login_pageviews",
        "avg_post_login_strict_contacts",
        "sessions_first_category_same_as_contact_category",
        "same_first_category_among_contact_sessions_rate",
        "sessions_first_city_same_as_contact_city",
        "same_first_city_among_contact_sessions_rate",
        "sessions_contacted_same_item_seen_prelogin",
        "contacted_same_item_seen_prelogin_session_rate",
        "sessions_contacted_same_category_seen_prelogin",
        "contacted_same_category_seen_prelogin_session_rate",
    ])

    df = add_pct_cols(df, [
        "strict_contact_after_login_session_rate",
        "same_first_category_among_contact_sessions_rate",
        "same_first_city_among_contact_sessions_rate",
        "contacted_same_item_seen_prelogin_session_rate",
        "contacted_same_category_seen_prelogin_session_rate",
    ])

    df["entry_label"] = make_short_label(
        df,
        ["first_nonlogin_category", "first_nonlogin_city", "first_nonlogin_surface"],
    )

    df["contact_label"] = make_short_label(
        df,
        ["first_contact_after_login_event_type", "first_contact_after_login_category"],
    )

    # 5.1 top entry paths
    plot_df = df.sort_values("sampled_sessions", ascending=False).head(TOP_N)

    fig = px.bar(
        plotly_safe_df(plot_df),
        x="entry_label",
        y="sampled_sessions",
        color="first_contact_after_login_event_type",
        hover_data=[
            "first_nonlogin_device",
            "first_contact_after_login_category",
            "first_contact_after_login_city",
            "sessions_with_strict_contact_after_login",
            "strict_contact_after_login_session_rate_pct",
            "avg_pre_nonlogin_events",
            "avg_post_login_events",
            "contacted_same_item_seen_prelogin_session_rate_pct",
            "contacted_same_category_seen_prelogin_session_rate_pct",
        ],
        title="Top non-login entry paths → first contact after login",
    )
    fig.update_layout(
        xaxis_title="First non-login category | city | surface",
        yaxis_title="Sampled sessions",
        xaxis_tickangle=-35,
        height=750,
    )
    save_fig(fig, "09_top_entry_paths_to_contact")

    # 5.2 aggregate by first non-login category
    by_cat = (
        df.groupby("first_nonlogin_category", as_index=False)
        .agg(
            sampled_sessions=("sampled_sessions", "sum"),
            sessions_with_strict_contact_after_login=("sessions_with_strict_contact_after_login", "sum"),
            sessions_first_category_same_as_contact_category=("sessions_first_category_same_as_contact_category", "sum"),
            sessions_first_city_same_as_contact_city=("sessions_first_city_same_as_contact_city", "sum"),
            sessions_contacted_same_item_seen_prelogin=("sessions_contacted_same_item_seen_prelogin", "sum"),
            sessions_contacted_same_category_seen_prelogin=("sessions_contacted_same_category_seen_prelogin", "sum"),
            avg_pre_nonlogin_events=("avg_pre_nonlogin_events", "mean"),
            avg_post_login_events=("avg_post_login_events", "mean"),
        )
    )

    by_cat["strict_contact_after_login_session_rate"] = weighted_rate(
        by_cat["sessions_with_strict_contact_after_login"],
        by_cat["sampled_sessions"]
    )
    by_cat["same_first_category_among_contact_sessions_rate"] = weighted_rate(
        by_cat["sessions_first_category_same_as_contact_category"],
        by_cat["sessions_with_strict_contact_after_login"]
    )
    by_cat["same_first_city_among_contact_sessions_rate"] = weighted_rate(
        by_cat["sessions_first_city_same_as_contact_city"],
        by_cat["sessions_with_strict_contact_after_login"]
    )
    by_cat["contacted_same_item_seen_prelogin_session_rate"] = weighted_rate(
        by_cat["sessions_contacted_same_item_seen_prelogin"],
        by_cat["sampled_sessions"]
    )
    by_cat["contacted_same_category_seen_prelogin_session_rate"] = weighted_rate(
        by_cat["sessions_contacted_same_category_seen_prelogin"],
        by_cat["sampled_sessions"]
    )

    by_cat = add_pct_cols(by_cat, [
        "strict_contact_after_login_session_rate",
        "same_first_category_among_contact_sessions_rate",
        "same_first_city_among_contact_sessions_rate",
        "contacted_same_item_seen_prelogin_session_rate",
        "contacted_same_category_seen_prelogin_session_rate",
    ])

    by_cat = by_cat.sort_values("sampled_sessions", ascending=False)
    save_derived(by_cat, "path_by_first_nonlogin_category")

    fig = px.bar(
        plotly_safe_df(by_cat),
        x="first_nonlogin_category",
        y="sampled_sessions",
        hover_data=[
            "sessions_with_strict_contact_after_login",
            "strict_contact_after_login_session_rate_pct",
            "contacted_same_item_seen_prelogin_session_rate_pct",
            "contacted_same_category_seen_prelogin_session_rate_pct",
        ],
        title="Non-login → login sessions by first non-login category",
    )
    fig.update_layout(
        xaxis_title="First non-login category",
        yaxis_title="Sampled sessions",
        xaxis_tickangle=-25,
        height=600,
    )
    save_fig(fig, "10_entry_category_volume")

    # 5.3 contact rate by first non-login category
    plot_df = by_cat[by_cat["sampled_sessions"] >= MIN_SESSIONS_FOR_RATE].copy()
    plot_df = plot_df.sort_values("strict_contact_after_login_session_rate_pct", ascending=False)

    fig = px.bar(
        plotly_safe_df(plot_df),
        x="first_nonlogin_category",
        y="strict_contact_after_login_session_rate_pct",
        text="strict_contact_after_login_session_rate_pct",
        hover_data=[
            "sampled_sessions",
            "sessions_with_strict_contact_after_login",
            "contacted_same_item_seen_prelogin_session_rate_pct",
            "contacted_same_category_seen_prelogin_session_rate_pct",
        ],
        title=f"Strict contact after login rate by first non-login category<br><sup>Only groups with sampled_sessions ≥ {MIN_SESSIONS_FOR_RATE}</sup>",
    )
    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        xaxis_title="First non-login category",
        yaxis_title="Strict contact after login rate (%)",
        xaxis_tickangle=-25,
        height=600,
    )
    save_fig(fig, "11_entry_category_contact_rate")

    # 5.4 contact event type distribution
    contact_df = df[
        df["first_contact_after_login_event_type"].astype(str) != "no_strict_contact_after_login"
    ].copy()

    if len(contact_df) > 0:
        by_contact_event = (
            contact_df.groupby("first_contact_after_login_event_type", as_index=False)
            .agg(
                sampled_sessions=("sampled_sessions", "sum"),
                sessions_with_strict_contact_after_login=("sessions_with_strict_contact_after_login", "sum"),
            )
            .sort_values("sessions_with_strict_contact_after_login", ascending=False)
        )

        fig = px.bar(
            plotly_safe_df(by_contact_event),
            x="first_contact_after_login_event_type",
            y="sessions_with_strict_contact_after_login",
            text="sessions_with_strict_contact_after_login",
            title="First strict contact event after login",
        )
        fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
        fig.update_layout(
            xaxis_title="First contact event type after login",
            yaxis_title="Sessions with strict contact",
            height=560,
        )
        save_fig(fig, "12_first_contact_event_after_login")

        by_contact_cat = (
            contact_df.groupby("first_contact_after_login_category", as_index=False)
            .agg(
                sessions_with_strict_contact_after_login=("sessions_with_strict_contact_after_login", "sum"),
                sampled_sessions=("sampled_sessions", "sum"),
            )
            .sort_values("sessions_with_strict_contact_after_login", ascending=False)
        )

        fig = px.bar(
            plotly_safe_df(by_contact_cat),
            x="first_contact_after_login_category",
            y="sessions_with_strict_contact_after_login",
            text="sessions_with_strict_contact_after_login",
            title="First strict contact category after login",
        )
        fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
        fig.update_layout(
            xaxis_title="First contact category after login",
            yaxis_title="Sessions with strict contact",
            xaxis_tickangle=-25,
            height=560,
        )
        save_fig(fig, "13_first_contact_category_after_login")

        # 5.5 Sankey: first non-login category -> first contact category
        sankey_df = (
            contact_df.groupby(
                ["first_nonlogin_category", "first_contact_after_login_category"],
                as_index=False
            )
            .agg(value=("sessions_with_strict_contact_after_login", "sum"))
            .sort_values("value", ascending=False)
            .head(30)
        )

        labels = (
            list(sankey_df["first_nonlogin_category"].astype(str).unique())
            + list(sankey_df["first_contact_after_login_category"].astype(str).unique())
        )
        labels = list(dict.fromkeys(labels))
        label_idx = {lab: i for i, lab in enumerate(labels)}

        fig = go.Figure(data=[go.Sankey(
            node=dict(
                pad=15,
                thickness=16,
                line=dict(width=0.5),
                label=labels,
            ),
            link=dict(
                source=[label_idx[x] for x in sankey_df["first_nonlogin_category"].astype(str)],
                target=[label_idx[x] for x in sankey_df["first_contact_after_login_category"].astype(str)],
                value=sankey_df["value"].tolist(),
            )
        )])

        fig.update_layout(
            title="Sankey: first non-login category → first contact category after login",
            height=650,
        )
        save_fig(fig, "14_sankey_entry_category_to_contact_category")

    # 5.6 same item/category after login metrics by first category
    plot_df = by_cat.copy()
    fig = px.bar(
        plotly_safe_df(plot_df),
        x="first_nonlogin_category",
        y=[
            "contacted_same_item_seen_prelogin_session_rate_pct",
            "contacted_same_category_seen_prelogin_session_rate_pct",
            "same_first_category_among_contact_sessions_rate_pct",
        ],
        barmode="group",
        hover_data=[
            "sampled_sessions",
            "sessions_with_strict_contact_after_login",
        ],
        title="Does pre-login intent transfer into post-login contact?",
    )
    fig.update_layout(
        xaxis_title="First non-login category",
        yaxis_title="Rate (%)",
        xaxis_tickangle=-25,
        height=620,
    )
    save_fig(fig, "15_prelogin_intent_transfer_by_category")

    # 5.7 device/surface
    by_device = (
        df.groupby(["first_nonlogin_device", "first_nonlogin_surface"], as_index=False)
        .agg(
            sampled_sessions=("sampled_sessions", "sum"),
            sessions_with_strict_contact_after_login=("sessions_with_strict_contact_after_login", "sum"),
            sessions_contacted_same_item_seen_prelogin=("sessions_contacted_same_item_seen_prelogin", "sum"),
            sessions_contacted_same_category_seen_prelogin=("sessions_contacted_same_category_seen_prelogin", "sum"),
        )
    )
    by_device["strict_contact_after_login_session_rate"] = weighted_rate(
        by_device["sessions_with_strict_contact_after_login"],
        by_device["sampled_sessions"]
    )
    by_device["same_item_contact_rate"] = weighted_rate(
        by_device["sessions_contacted_same_item_seen_prelogin"],
        by_device["sampled_sessions"]
    )
    by_device["same_category_contact_rate"] = weighted_rate(
        by_device["sessions_contacted_same_category_seen_prelogin"],
        by_device["sampled_sessions"]
    )
    by_device = add_pct_cols(by_device, [
        "strict_contact_after_login_session_rate",
        "same_item_contact_rate",
        "same_category_contact_rate",
    ])
    by_device["label"] = by_device["first_nonlogin_device"].astype(str) + " | " + by_device["first_nonlogin_surface"].astype(str)
    by_device = by_device.sort_values("sampled_sessions", ascending=False)

    fig = px.bar(
        plotly_safe_df(by_device.head(TOP_N)),
        x="label",
        y="sampled_sessions",
        color="first_nonlogin_device",
        hover_data=[
            "sessions_with_strict_contact_after_login",
            "strict_contact_after_login_session_rate_pct",
            "same_item_contact_rate_pct",
            "same_category_contact_rate_pct",
        ],
        title="Entry device × surface volume",
    )
    fig.update_layout(
        xaxis_title="Device | surface",
        yaxis_title="Sampled sessions",
        xaxis_tickangle=-35,
        height=650,
    )
    save_fig(fig, "16_entry_device_surface_volume")

# ============================================================
# 6. TABLE 3 — SAME SESSION TRANSFER
# ============================================================

if transfer is not None and len(transfer) > 0:
    df = transfer.copy()

    df = to_num(df, [
        "sampled_sessions",
        "sessions_with_post_login_event",
        "sessions_with_strict_contact_after_login",
        "strict_contact_after_login_session_rate",
        "avg_prelogin_view_items",
        "avg_prelogin_view_categories",
        "sessions_same_item_reappears_after_login",
        "same_item_reappears_after_login_rate",
        "sessions_same_item_contact_after_login",
        "same_item_contact_after_login_rate",
        "sessions_same_category_reappears_after_login",
        "same_category_reappears_after_login_rate",
        "sessions_same_category_contact_after_login",
        "same_category_contact_after_login_rate",
        "avg_pre_items_reappear_after_login",
        "avg_pre_items_contacted_after_login",
        "avg_pre_categories_reappear_after_login",
        "avg_pre_categories_contacted_after_login",
    ])

    df = add_pct_cols(df, [
        "strict_contact_after_login_session_rate",
        "same_item_reappears_after_login_rate",
        "same_item_contact_after_login_rate",
        "same_category_reappears_after_login_rate",
        "same_category_contact_after_login_rate",
    ])

    df["entry_label"] = make_short_label(
        df,
        ["first_nonlogin_category", "first_nonlogin_city", "first_nonlogin_surface"],
    )

    # 6.1 top transfer paths by volume
    plot_df = df.sort_values("sampled_sessions", ascending=False).head(TOP_N)

    fig = px.bar(
        plotly_safe_df(plot_df),
        x="entry_label",
        y="sampled_sessions",
        color="first_nonlogin_category",
        hover_data=[
            "first_nonlogin_device",
            "strict_contact_after_login_session_rate_pct",
            "avg_prelogin_view_items",
            "avg_prelogin_view_categories",
            "same_item_reappears_after_login_rate_pct",
            "same_item_contact_after_login_rate_pct",
            "same_category_reappears_after_login_rate_pct",
            "same_category_contact_after_login_rate_pct",
        ],
        title="Same-session transfer: top non-login entry groups",
    )
    fig.update_layout(
        xaxis_title="First non-login category | city | surface",
        yaxis_title="Sampled sessions",
        xaxis_tickangle=-35,
        height=720,
    )
    save_fig(fig, "17_transfer_top_entry_groups")

    # 6.2 aggregate transfer by first non-login category
    by_cat = (
        df.groupby("first_nonlogin_category", as_index=False)
        .agg(
            sampled_sessions=("sampled_sessions", "sum"),
            sessions_with_strict_contact_after_login=("sessions_with_strict_contact_after_login", "sum"),
            sessions_same_item_reappears_after_login=("sessions_same_item_reappears_after_login", "sum"),
            sessions_same_item_contact_after_login=("sessions_same_item_contact_after_login", "sum"),
            sessions_same_category_reappears_after_login=("sessions_same_category_reappears_after_login", "sum"),
            sessions_same_category_contact_after_login=("sessions_same_category_contact_after_login", "sum"),
            avg_prelogin_view_items=("avg_prelogin_view_items", "mean"),
            avg_prelogin_view_categories=("avg_prelogin_view_categories", "mean"),
        )
    )

    by_cat["strict_contact_after_login_session_rate"] = weighted_rate(
        by_cat["sessions_with_strict_contact_after_login"],
        by_cat["sampled_sessions"]
    )
    by_cat["same_item_reappears_after_login_rate"] = weighted_rate(
        by_cat["sessions_same_item_reappears_after_login"],
        by_cat["sampled_sessions"]
    )
    by_cat["same_item_contact_after_login_rate"] = weighted_rate(
        by_cat["sessions_same_item_contact_after_login"],
        by_cat["sampled_sessions"]
    )
    by_cat["same_category_reappears_after_login_rate"] = weighted_rate(
        by_cat["sessions_same_category_reappears_after_login"],
        by_cat["sampled_sessions"]
    )
    by_cat["same_category_contact_after_login_rate"] = weighted_rate(
        by_cat["sessions_same_category_contact_after_login"],
        by_cat["sampled_sessions"]
    )

    by_cat = add_pct_cols(by_cat, [
        "strict_contact_after_login_session_rate",
        "same_item_reappears_after_login_rate",
        "same_item_contact_after_login_rate",
        "same_category_reappears_after_login_rate",
        "same_category_contact_after_login_rate",
    ])
    by_cat = by_cat.sort_values("sampled_sessions", ascending=False)
    save_derived(by_cat, "transfer_by_first_nonlogin_category")

    fig = px.bar(
        plotly_safe_df(by_cat),
        x="first_nonlogin_category",
        y=[
            "same_item_reappears_after_login_rate_pct",
            "same_item_contact_after_login_rate_pct",
            "same_category_reappears_after_login_rate_pct",
            "same_category_contact_after_login_rate_pct",
        ],
        barmode="group",
        hover_data=[
            "sampled_sessions",
            "strict_contact_after_login_session_rate_pct",
            "avg_prelogin_view_items",
            "avg_prelogin_view_categories",
        ],
        title="Same-session transfer rates by first non-login category",
    )
    fig.update_layout(
        xaxis_title="First non-login category",
        yaxis_title="Rate (%)",
        xaxis_tickangle=-25,
        height=650,
    )
    save_fig(fig, "18_transfer_rates_by_category")

    # 6.3 scatter: prelogin depth vs same-item contact
    plot_df = df[df["sampled_sessions"] >= MIN_SESSIONS_FOR_RATE].copy()
    fig = px.scatter(
        plotly_safe_df(plot_df),
        x="avg_prelogin_view_items",
        y="same_item_contact_after_login_rate_pct",
        size="sampled_sessions",
        color="first_nonlogin_category",
        hover_name="entry_label",
        hover_data=[
            "first_nonlogin_city",
            "first_nonlogin_surface",
            "first_nonlogin_device",
            "strict_contact_after_login_session_rate_pct",
            "same_category_contact_after_login_rate_pct",
        ],
        title=f"Pre-login browsing depth vs same-item contact after login<br><sup>Only groups with sampled_sessions ≥ {MIN_SESSIONS_FOR_RATE}</sup>",
    )
    fig.update_layout(
        xaxis_title="Avg pre-login viewed items",
        yaxis_title="Same item contacted after login (%)",
        height=650,
    )
    save_fig(fig, "19_prelogin_depth_vs_same_item_contact")

    # 6.4 city-category heatmap for same category contact
    top_cities = (
        df.groupby("first_nonlogin_city", as_index=False)["sampled_sessions"]
        .sum()
        .sort_values("sampled_sessions", ascending=False)
        .head(15)["first_nonlogin_city"]
        .tolist()
    )

    heat_df = df[df["first_nonlogin_city"].isin(top_cities)].copy()

    heat = (
        heat_df.pivot_table(
            index="first_nonlogin_city",
            columns="first_nonlogin_category",
            values="same_category_contact_after_login_rate_pct",
            aggfunc="mean",
        )
        .fillna(0)
    )

    fig = px.imshow(
        heat,
        text_auto=".2f",
        aspect="auto",
        title="Same-category contact after login rate: city × first non-login category (%)",
        labels=dict(x="First non-login category", y="First non-login city", color="Rate (%)"),
    )
    fig.update_layout(height=650)
    save_fig(fig, "20_heatmap_same_category_contact_rate_city_category")

# ============================================================
# 7. SAVE CHART INDEX
# ============================================================

chart_index_df = pd.DataFrame(chart_index)
chart_index_path = os.path.join(OUTPUT_DIR, "99_chart_index.csv")
chart_index_df.to_csv(chart_index_path, index=False, encoding="utf-8-sig")

print("\nDONE.")
print("Figures folder:", FIG_DIR)
print("Derived tables:", TABLE_OUT_DIR)
print("Chart index:", chart_index_path)
display(chart_index_df)

print("""
NÊN MỞ TRƯỚC MẤY CHART NÀY:

00_overall_kpi_cards.html
03_tet_avg_daily_event_rows.html
05_tet_strict_contact_per_pageview.html
09_top_entry_paths_to_contact.html
11_entry_category_contact_rate.html
14_sankey_entry_category_to_contact_category.html
15_prelogin_intent_transfer_by_category.html
18_transfer_rates_by_category.html
19_prelogin_depth_vs_same_item_contact.html
""")

OUTPUT_DIR: /kaggle/working/eda_non_login_extra_3_tables_visuals
[LOAD] 00_session_gate_summary_sample | rows: 4 | /kaggle/working/eda_non_login_extra_3_tables/tables/00_session_gate_summary_sample.parquet
[LOAD] 01_nonlogin_entry_to_login_contact_path_summary | rows: 300 | /kaggle/working/eda_non_login_extra_3_tables/tables/01_nonlogin_entry_to_login_contact_path_summary.parquet
[LOAD] 02_tet_impact_strict_contact_window_summary | rows: 6 | /kaggle/working/eda_non_login_extra_3_tables/tables/02_tet_impact_strict_contact_window_summary.parquet
[LOAD] 02A_daily_strict_contact_rate_full | rows: 304 | /kaggle/working/eda_non_login_extra_3_tables/tables/02A_daily_strict_contact_rate_full.parquet
[LOAD] 03_same_session_prelogin_transfer_summary | rows: 300 | /kaggle/working/eda_non_login_extra_3_tables/tables/03_same_session_prelogin_transfer_summary.parquet
[LOAD] 04_overall_nonlogin_to_login_transfer_summary | rows: 1 | /kaggle/working/eda_non_login_extra_3_tables/tables/04_overall_nonlog

[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/00b_overall_summary_table.html


[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/01_session_gate_volume.html


[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/02_session_gate_strict_contact_rate.html


[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/03_tet_avg_daily_event_rows.html


...

[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/17_transfer_top_entry_groups.html


[SAVE DERIVED] /kaggle/working/eda_non_login_extra_3_tables_visuals/derived_tables/transfer_by_first_nonlogin_category.csv


,first_nonlogin_category,sampled_sessions,sessions_with_strict_contact_after_login,sessions_same_item_reappears_after_login,sessions_same_item_contact_after_login,sessions_same_category_reappears_after_login,sessions_same_category_contact_after_login,avg_prelogin_view_items,avg_prelogin_view_categories,strict_contact_after_login_session_rate,same_item_reappears_after_login_rate,same_item_contact_after_login_rate,same_category_reappears_after_login_rate,same_category_contact_after_login_rate,strict_contact_after_login_session_rate_pct,same_item_reappears_after_login_rate_pct,same_item_contact_after_login_rate_pct,same_category_reappears_after_login_rate_pct,same_category_contact_after_login_rate_pct
1,1020_apartment,7757,1755.0,7.0,1.0,4715.0,876.0,1.822685,1.122292,0.226247,0.000902,0.000129,0.607838,0.112930,22.624726,0.090241,0.012892,60.783808,11.293026
4,1050_new_project,4329,989.0,12.0,1.0,1581.0,301.0,1.900709,1.130380,0.228459,0.002772,0.000231,0.365211,0.069531,22.845923,0.277200,0.023100,36.521137,6.953107
0,1010_room_rental,3860,836.0,6.0,2.0,1183.0,203.0,1.590989,1.040494,0.216580,0.001554,0.000518,0.306477,0.052591,21.658031,0.155440,0.051813,30.647668,5.259067
3,1040_land_commercial,2116,479.0,2.0,2.0,656.0,89.0,1.700475,1.143293,0.226371,0.000945,0.000945,0.310019,0.042060,22.637051,0.094518,0.094518,31.001890,4.206049
2,1030_house,1008,228.0,1.0,0.0,227.0,37.0,1.544149,1.055750,0.226190,0.000992,0.000000,0.225198,0.036706,22.619048,0.099206,0.000000,22.519841,3.670635


[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/18_transfer_rates_by_category.html


ValueError: 
    Invalid value of type 'pandas.core.series.Series' received for the 'size' property of scatter.marker
        Received value: 0     3211
3     1585
6      533
10     264
13     230
14     226
15     189
16     174
24     127
25     117
26     117
29      91
36      75
38      70
48      45
50      44
51      41
57      33
62      31
63      31
70      24
72      22
73      22
76      20
Name: sampled_sessions, dtype: object

    The 'size' property is a number and may be specified as:
      - An int or float in the interval [0, inf]
      - A tuple, list, or one-dimensional numpy array of the above

In [6]:
# ============================================================
# FIX CHART 19 + CONTINUE CHART 20 + SAVE INDEX
# Run this after the Plotly size error
# ============================================================

import os
import glob
import pandas as pd
import numpy as np
import plotly.express as px

# Nếu các biến này đã có từ cell trước thì dùng lại.
# Nếu không có, đọc lại từ output table.
OUTPUT_DIR = "/kaggle/working/eda_non_login_extra_3_tables_visuals"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLE_OUT_DIR = os.path.join(OUTPUT_DIR, "derived_tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_OUT_DIR, exist_ok=True)

MIN_SESSIONS_FOR_RATE = 20

def find_table(stem):
    roots = [
        "/kaggle/working/eda_non_login_extra_3_tables/tables",
        "/kaggle/working/eda_non_login_extra_3_tables_visuals/derived_tables",
        "/kaggle/input",
    ]
    paths = []
    for root in roots:
        if os.path.exists(root):
            paths += glob.glob(os.path.join(root, "**", f"{stem}.parquet"), recursive=True)
            paths += glob.glob(os.path.join(root, "**", f"{stem}.csv"), recursive=True)

    paths = sorted(set(paths))
    if len(paths) == 0:
        raise FileNotFoundError(f"Cannot find table: {stem}")

    paths = sorted(paths, key=lambda p: 0 if p.endswith(".parquet") else 1)
    return paths[0]

def read_table(stem):
    path = find_table(stem)
    print("[LOAD]", stem, path)
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    return pd.read_csv(path)

def to_num(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def save_fig(fig, name, title=None):
    html_path = os.path.join(FIG_DIR, f"{name}.html")
    fig.write_html(html_path)
    print("[SAVE FIG]", html_path)
    fig.show()

# Load transfer table nếu biến transfer không còn
try:
    transfer
except NameError:
    transfer = read_table("03_same_session_prelogin_transfer_summary")

df = transfer.copy()

# Ép numeric lại cho chắc
numeric_cols = [
    "sampled_sessions",
    "strict_contact_after_login_session_rate",
    "avg_prelogin_view_items",
    "avg_prelogin_view_categories",
    "same_item_reappears_after_login_rate",
    "same_item_contact_after_login_rate",
    "same_category_reappears_after_login_rate",
    "same_category_contact_after_login_rate",
]

df = to_num(df, numeric_cols)

# Tạo pct cols nếu chưa có
for c in [
    "strict_contact_after_login_session_rate",
    "same_item_reappears_after_login_rate",
    "same_item_contact_after_login_rate",
    "same_category_reappears_after_login_rate",
    "same_category_contact_after_login_rate",
]:
    pct_col = c + "_pct"
    if pct_col not in df.columns and c in df.columns:
        df[pct_col] = df[c] * 100

df["entry_label"] = (
    df["first_nonlogin_category"].astype(str)
    + " | "
    + df["first_nonlogin_city"].astype(str)
    + " | "
    + df["first_nonlogin_surface"].astype(str)
)

# ============================================================
# FIXED CHART 19
# Không dùng plotly_safe_df ở scatter vì size phải numeric
# ============================================================

plot_df = df[df["sampled_sessions"] >= MIN_SESSIONS_FOR_RATE].copy()

plot_df = plot_df.dropna(subset=[
    "avg_prelogin_view_items",
    "same_item_contact_after_login_rate_pct",
    "sampled_sessions",
])

# Đảm bảo size numeric thật sự
plot_df["sampled_sessions"] = pd.to_numeric(plot_df["sampled_sessions"], errors="coerce")
plot_df = plot_df[plot_df["sampled_sessions"] > 0].copy()

fig = px.scatter(
    plot_df,
    x="avg_prelogin_view_items",
    y="same_item_contact_after_login_rate_pct",
    size="sampled_sessions",
    size_max=55,
    color="first_nonlogin_category",
    hover_name="entry_label",
    hover_data=[
        "first_nonlogin_city",
        "first_nonlogin_surface",
        "first_nonlogin_device",
        "sampled_sessions",
        "strict_contact_after_login_session_rate_pct",
        "same_category_contact_after_login_rate_pct",
    ],
    title=f"Pre-login browsing depth vs same-item contact after login<br><sup>Only groups with sampled_sessions ≥ {MIN_SESSIONS_FOR_RATE}</sup>",
)

fig.update_layout(
    xaxis_title="Avg pre-login viewed items",
    yaxis_title="Same item contacted after login (%)",
    height=650,
)

save_fig(fig, "19_prelogin_depth_vs_same_item_contact")

# ============================================================
# CHART 20 — heatmap same-category contact rate
# ============================================================

top_cities = (
    df.groupby("first_nonlogin_city", as_index=False)["sampled_sessions"]
    .sum()
    .sort_values("sampled_sessions", ascending=False)
    .head(15)["first_nonlogin_city"]
    .tolist()
)

heat_df = df[df["first_nonlogin_city"].isin(top_cities)].copy()

heat = (
    heat_df.pivot_table(
        index="first_nonlogin_city",
        columns="first_nonlogin_category",
        values="same_category_contact_after_login_rate_pct",
        aggfunc="mean",
    )
    .fillna(0)
)

fig = px.imshow(
    heat,
    text_auto=".2f",
    aspect="auto",
    title="Same-category contact after login rate: city × first non-login category (%)",
    labels=dict(
        x="First non-login category",
        y="First non-login city",
        color="Rate (%)"
    ),
)

fig.update_layout(height=650)
save_fig(fig, "20_heatmap_same_category_contact_rate_city_category")

print("\nDONE fixed charts 19 and 20.")
print("Figures folder:", FIG_DIR)

[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/19_prelogin_depth_vs_same_item_contact.html


[SAVE FIG] /kaggle/working/eda_non_login_extra_3_tables_visuals/figures/20_heatmap_same_category_contact_rate_city_category.html



DONE fixed charts 19 and 20.
Figures folder: /kaggle/working/eda_non_login_extra_3_tables_visuals/figures


In [1]:
# ============================================================
# ADD-ON EDA:
# USER SHARE vs STRICT CONTACT CONTRIBUTION BY BEHAVIOR SEGMENT
#
# Goal:
#   Kiểm tra:
#   - Segment nào chiếm bao nhiêu % users?
#   - Segment đó đóng góp bao nhiêu % strict contact?
#   - focused_demand có nhiều user nhưng có thật sự tạo nhiều contact không?
#   - broker-like/broad_searcher ít user nhưng có over-contribute contact không?
#
# Strict contact = view_phone + contact_chat + contact_zalo + contact_sms
# Không tính other_interaction.
#
# Input expected:
#   user_positive_category_ad_profile
#   hoặc file:
#   /kaggle/working/eda_category_adtype_broker_like/user_positive_category_ad_profile_lite.parquet
#
# Output:
#   08_segment_user_vs_strict_contact_contribution.csv
#   09_segment_strict_contact_productivity.csv
#   10_dominant_combo_strict_contact_contribution.csv
#   11_segment_contact_intensity_quantiles.csv
#   PNG charts in /segment_strict_contact_addon/
# ============================================================

import os
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from IPython.display import display

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

import matplotlib.pyplot as plt

# ============================================================
# 0. CONFIG
# ============================================================

BASE_OUTPUT_DIR = "/kaggle/working/eda_category_adtype_broker_like"
ADDON_DIR = os.path.join(BASE_OUTPUT_DIR, "segment_strict_contact_addon")
os.makedirs(ADDON_DIR, exist_ok=True)

DB_PATH = os.path.join(BASE_OUTPUT_DIR, "category_adtype_broker_like.duckdb")
PROFILE_PARQUET = os.path.join(BASE_OUTPUT_DIR, "user_positive_category_ad_profile_lite.parquet")

# Nếu bạn lưu ở chỗ khác thì chỉnh 2 biến trên.

SEGMENT_ORDER = [
    "focused_demand",
    "broad_searcher",
    "multi_market_broker_like",
    "rental_broker_like",
    "land_project_speculator_like",
]

print("DB_PATH:", DB_PATH)
print("PROFILE_PARQUET:", PROFILE_PARQUET)
print("ADDON_DIR:", ADDON_DIR)

# ============================================================
# 1. OPEN DATA
# ============================================================

if os.path.exists(DB_PATH):
    con = duckdb.connect(DB_PATH)
else:
    con = duckdb.connect()

# Nếu table không còn trong DB, load từ parquet đã save.
tables = set(con.execute("SHOW TABLES").df()["name"].tolist())

if "user_positive_category_ad_profile" not in tables:
    if not os.path.exists(PROFILE_PARQUET):
        raise FileNotFoundError(
            "Không thấy table user_positive_category_ad_profile trong DB "
            "và cũng không thấy user_positive_category_ad_profile_lite.parquet. "
            "Bạn cần chạy xong code LITE trước."
        )

    con.execute(f"""
    CREATE OR REPLACE TABLE user_positive_category_ad_profile AS
    SELECT *
    FROM read_parquet('{PROFILE_PARQUET.replace("\\", "/").replace("'", "''")}')
    """)

print("Loaded table: user_positive_category_ad_profile")

cols = set(
    con.execute("DESCRIBE user_positive_category_ad_profile").df()["column_name"].tolist()
)

required_cols = ["user_id", "behavior_segment"]
for c in required_cols:
    if c not in cols:
        raise ValueError(f"Missing required column: {c}")

def zero_if_missing(colname):
    if colname in cols:
        return f"COALESCE({colname}, 0)"
    return "0"

view_phone_expr = zero_if_missing("view_phone_events_known")
chat_expr = zero_if_missing("contact_chat_events_known")
zalo_expr = zero_if_missing("contact_zalo_events_known")
sms_expr = zero_if_missing("contact_sms_events_known")
other_expr = zero_if_missing("other_interaction_events_known")
positive_expr = zero_if_missing("positive_events_known")

strict_expr = f"""
(
    {view_phone_expr}
    + {chat_expr}
    + {zalo_expr}
    + {sms_expr}
)
"""

# ============================================================
# 2. BUILD CLEAN METRIC VIEW
# ============================================================

con.execute(f"""
CREATE OR REPLACE TABLE segment_contact_base AS
SELECT
    user_id,
    behavior_segment,

    {positive_expr} AS positive_events_including_other,

    {strict_expr} AS strict_contact_events,

    {view_phone_expr} AS view_phone_events,
    {chat_expr} AS contact_chat_events,
    {zalo_expr} AS contact_zalo_events,
    {sms_expr} AS contact_sms_events,
    {other_expr} AS other_interaction_events,

    CASE WHEN {strict_expr} > 0 THEN 1 ELSE 0 END AS has_strict_contact,

    CASE
        WHEN {positive_expr} > 0
        THEN ({strict_expr}) * 1.0 / NULLIF({positive_expr}, 0)
        ELSE NULL
    END AS strict_share_within_positive,

    dominant_combo_label,
    dominant_category_label,
    dominant_ad_type_label,
    dominant_combo_share,

    broker_like_score,
    is_broker_like_candidate,

    approx_positive_items,
    approx_distinct_sellers_combo_sum,
    approx_distinct_cities_combo_sum,
    approx_distinct_districts_combo_sum,
    distinct_categories,
    distinct_ad_types,
    distinct_category_ad_combos

FROM user_positive_category_ad_profile
""")

check_df = con.execute("""
SELECT
    COUNT(*) AS users,
    SUM(positive_events_including_other) AS positive_events_including_other,
    SUM(strict_contact_events) AS strict_contact_events,
    SUM(other_interaction_events) AS other_interaction_events,
    SUM(strict_contact_events) * 1.0 / NULLIF(SUM(positive_events_including_other), 0)
        AS strict_share_of_all_positive_events
FROM segment_contact_base
""").df()

display(check_df)
check_df.to_csv(
    os.path.join(ADDON_DIR, "00_strict_contact_check.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 3. SEGMENT CONTRIBUTION TABLE
# ============================================================

segment_contribution = con.execute("""
WITH seg AS (
    SELECT
        behavior_segment,

        COUNT(*) AS users,
        SUM(has_strict_contact) AS users_with_strict_contact,

        SUM(positive_events_including_other) AS positive_events_including_other,
        SUM(strict_contact_events) AS strict_contact_events,
        SUM(other_interaction_events) AS other_interaction_events,

        SUM(view_phone_events) AS view_phone_events,
        SUM(contact_chat_events) AS contact_chat_events,
        SUM(contact_zalo_events) AS contact_zalo_events,
        SUM(contact_sms_events) AS contact_sms_events,

        AVG(strict_contact_events) AS avg_strict_contacts_per_user,
        MEDIAN(strict_contact_events) AS median_strict_contacts_per_user,

        AVG(strict_share_within_positive) AS avg_strict_share_within_positive,

        AVG(approx_distinct_sellers_combo_sum) AS avg_approx_distinct_sellers,
        AVG(approx_distinct_districts_combo_sum) AS avg_approx_distinct_districts,
        AVG(distinct_category_ad_combos) AS avg_distinct_category_ad_combos

    FROM segment_contact_base
    GROUP BY behavior_segment
),
total AS (
    SELECT
        SUM(users) AS total_users,
        SUM(users_with_strict_contact) AS total_users_with_strict_contact,
        SUM(positive_events_including_other) AS total_positive_events_including_other,
        SUM(strict_contact_events) AS total_strict_contact_events,
        SUM(other_interaction_events) AS total_other_interaction_events
    FROM seg
)
SELECT
    s.*,

    s.users * 1.0 / NULLIF(t.total_users, 0) AS user_share,

    s.users_with_strict_contact * 1.0 / NULLIF(t.total_users_with_strict_contact, 0)
        AS strict_contact_user_share,

    s.users_with_strict_contact * 1.0 / NULLIF(s.users, 0)
        AS strict_contact_user_rate_within_segment,

    s.positive_events_including_other * 1.0 / NULLIF(t.total_positive_events_including_other, 0)
        AS positive_event_share,

    s.strict_contact_events * 1.0 / NULLIF(t.total_strict_contact_events, 0)
        AS strict_contact_share,

    s.other_interaction_events * 1.0 / NULLIF(t.total_other_interaction_events, 0)
        AS other_interaction_share,

    s.strict_contact_events * 1.0 / NULLIF(s.users, 0)
        AS strict_contacts_per_user,

    s.strict_contact_events * 1.0 / NULLIF(s.users_with_strict_contact, 0)
        AS strict_contacts_per_contacting_user,

    s.strict_contact_events * 1.0 / NULLIF(s.positive_events_including_other, 0)
        AS strict_share_of_segment_positive,

    s.other_interaction_events * 1.0 / NULLIF(s.positive_events_including_other, 0)
        AS other_share_of_segment_positive,

    (
        s.strict_contact_events * 1.0 / NULLIF(t.total_strict_contact_events, 0)
    )
    / NULLIF(
        s.users * 1.0 / NULLIF(t.total_users, 0),
        0
    ) AS strict_contact_contribution_index,

    (
        s.strict_contact_events * 1.0 / NULLIF(t.total_strict_contact_events, 0)
    )
    -
    (
        s.users * 1.0 / NULLIF(t.total_users, 0)
    ) AS strict_share_minus_user_share,

    CASE
        WHEN (
            (
                s.strict_contact_events * 1.0 / NULLIF(t.total_strict_contact_events, 0)
            )
            / NULLIF(
                s.users * 1.0 / NULLIF(t.total_users, 0),
                0
            )
        ) >= 1.25
        THEN 'over-contributes strict contacts'

        WHEN (
            (
                s.strict_contact_events * 1.0 / NULLIF(t.total_strict_contact_events, 0)
            )
            / NULLIF(
                s.users * 1.0 / NULLIF(t.total_users, 0),
                0
            )
        ) <= 0.75
        THEN 'under-contributes strict contacts'

        ELSE 'roughly proportional'
    END AS contribution_interpretation

FROM seg s
CROSS JOIN total t
ORDER BY
    CASE s.behavior_segment
        WHEN 'focused_demand' THEN 1
        WHEN 'broad_searcher' THEN 2
        WHEN 'multi_market_broker_like' THEN 3
        WHEN 'rental_broker_like' THEN 4
        WHEN 'land_project_speculator_like' THEN 5
        ELSE 99
    END
""").df()

display(segment_contribution)

segment_contribution.to_csv(
    os.path.join(ADDON_DIR, "08_segment_user_vs_strict_contact_contribution.csv"),
    index=False,
    encoding="utf-8-sig"
)

# Bảng rút gọn để đưa vào slide
slide_table = segment_contribution[[
    "behavior_segment",
    "users",
    "user_share",
    "strict_contact_events",
    "strict_contact_share",
    "strict_contacts_per_user",
    "strict_contact_user_rate_within_segment",
    "strict_contact_contribution_index",
    "strict_share_minus_user_share",
    "contribution_interpretation",
]].copy()

display(slide_table)

slide_table.to_csv(
    os.path.join(ADDON_DIR, "09_segment_strict_contact_productivity.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 4. QUANTILES: SEGMENT CONTACT INTENSITY
# ============================================================

quantile_table = con.execute("""
SELECT
    behavior_segment,

    COUNT(*) AS users,

    quantile_cont(strict_contact_events, 0.00) AS q00_strict_contacts,
    quantile_cont(strict_contact_events, 0.25) AS q25_strict_contacts,
    quantile_cont(strict_contact_events, 0.50) AS q50_strict_contacts,
    quantile_cont(strict_contact_events, 0.75) AS q75_strict_contacts,
    quantile_cont(strict_contact_events, 0.90) AS q90_strict_contacts,
    quantile_cont(strict_contact_events, 0.95) AS q95_strict_contacts,
    quantile_cont(strict_contact_events, 0.99) AS q99_strict_contacts,
    MAX(strict_contact_events) AS max_strict_contacts,

    AVG(strict_contact_events) AS avg_strict_contacts

FROM segment_contact_base
GROUP BY behavior_segment
ORDER BY
    CASE behavior_segment
        WHEN 'focused_demand' THEN 1
        WHEN 'broad_searcher' THEN 2
        WHEN 'multi_market_broker_like' THEN 3
        WHEN 'rental_broker_like' THEN 4
        WHEN 'land_project_speculator_like' THEN 5
        ELSE 99
    END
""").df()

display(quantile_table)

quantile_table.to_csv(
    os.path.join(ADDON_DIR, "11_segment_contact_intensity_quantiles.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 5. DOMINANT COMBO CONTRIBUTION
# Nhìn trong mỗi behavior segment, combo nào tạo strict contact nhiều nhất.
# ============================================================

dominant_combo_contact = con.execute("""
WITH combo AS (
    SELECT
        behavior_segment,
        dominant_combo_label,
        dominant_category_label,
        dominant_ad_type_label,

        COUNT(*) AS users,
        SUM(strict_contact_events) AS strict_contact_events,
        SUM(positive_events_including_other) AS positive_events_including_other,

        AVG(strict_contact_events) AS avg_strict_contacts_per_user,
        SUM(has_strict_contact) * 1.0 / NULLIF(COUNT(*), 0) AS strict_contact_user_rate,

        AVG(approx_distinct_sellers_combo_sum) AS avg_approx_distinct_sellers,
        AVG(approx_distinct_districts_combo_sum) AS avg_approx_distinct_districts,
        AVG(distinct_category_ad_combos) AS avg_distinct_category_ad_combos

    FROM segment_contact_base
    GROUP BY
        behavior_segment,
        dominant_combo_label,
        dominant_category_label,
        dominant_ad_type_label
),
total AS (
    SELECT SUM(strict_contact_events) AS total_strict_contact_events
    FROM combo
),
ranked AS (
    SELECT
        c.*,

        c.strict_contact_events * 1.0 / NULLIF(t.total_strict_contact_events, 0)
            AS global_strict_contact_share,

        ROW_NUMBER() OVER (
            PARTITION BY c.behavior_segment
            ORDER BY c.strict_contact_events DESC, c.users DESC
        ) AS rn_in_segment

    FROM combo c
    CROSS JOIN total t
)
SELECT *
FROM ranked
WHERE rn_in_segment <= 20
ORDER BY
    behavior_segment,
    strict_contact_events DESC
""").df()

display(dominant_combo_contact)

dominant_combo_contact.to_csv(
    os.path.join(ADDON_DIR, "10_dominant_combo_strict_contact_contribution.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 6. STATIC CHARTS
# ============================================================

plot_df = segment_contribution.copy()

# Giữ order dễ đọc
plot_df["behavior_segment"] = pd.Categorical(
    plot_df["behavior_segment"],
    categories=[s for s in SEGMENT_ORDER if s in plot_df["behavior_segment"].unique()],
    ordered=True
)
plot_df = plot_df.sort_values("behavior_segment").reset_index(drop=True)

# ---------- Chart 1: User share vs strict contact share ----------
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(plot_df))
width = 0.36

ax.bar(
    x - width / 2,
    plot_df["user_share"],
    width,
    label="User share"
)

ax.bar(
    x + width / 2,
    plot_df["strict_contact_share"],
    width,
    label="Strict contact share"
)

ax.set_title("User share vs strict contact contribution by behavior segment")
ax.set_xlabel("Behavior segment")
ax.set_ylabel("Share")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["behavior_segment"], rotation=25, ha="right")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.legend()

for i, row in plot_df.iterrows():
    ax.text(
        i - width / 2,
        row["user_share"],
        f"{row['user_share']:.1%}",
        ha="center",
        va="bottom",
        fontsize=9
    )
    ax.text(
        i + width / 2,
        row["strict_contact_share"],
        f"{row['strict_contact_share']:.1%}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
chart_path = os.path.join(ADDON_DIR, "01_user_share_vs_strict_contact_share.png")
plt.savefig(chart_path, dpi=160)
plt.show()
print("[SAVE]", chart_path)

# ---------- Chart 2: Contribution index ----------
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    plot_df["behavior_segment"].astype(str),
    plot_df["strict_contact_contribution_index"]
)

ax.axhline(1.0, linestyle="--", linewidth=1)

ax.set_title("Strict contact contribution index by behavior segment")
ax.set_xlabel("Behavior segment")
ax.set_ylabel("Strict contact share / User share")
ax.tick_params(axis="x", rotation=25)

for i, row in plot_df.iterrows():
    ax.text(
        i,
        row["strict_contact_contribution_index"],
        f"{row['strict_contact_contribution_index']:.2f}x",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
chart_path = os.path.join(ADDON_DIR, "02_strict_contact_contribution_index.png")
plt.savefig(chart_path, dpi=160)
plt.show()
print("[SAVE]", chart_path)

# ---------- Chart 3: Strict contacts per user ----------
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    plot_df["behavior_segment"].astype(str),
    plot_df["strict_contacts_per_user"]
)

ax.set_title("Average strict contacts per user by behavior segment")
ax.set_xlabel("Behavior segment")
ax.set_ylabel("Strict contacts per user")
ax.tick_params(axis="x", rotation=25)

for i, row in plot_df.iterrows():
    ax.text(
        i,
        row["strict_contacts_per_user"],
        f"{row['strict_contacts_per_user']:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
chart_path = os.path.join(ADDON_DIR, "03_strict_contacts_per_user.png")
plt.savefig(chart_path, dpi=160)
plt.show()
print("[SAVE]", chart_path)

# ---------- Chart 4: Contact user rate ----------
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    plot_df["behavior_segment"].astype(str),
    plot_df["strict_contact_user_rate_within_segment"]
)

ax.set_title("Share of users with at least one strict contact")
ax.set_xlabel("Behavior segment")
ax.set_ylabel("Users with strict contact / users in segment")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.tick_params(axis="x", rotation=25)

for i, row in plot_df.iterrows():
    ax.text(
        i,
        row["strict_contact_user_rate_within_segment"],
        f"{row['strict_contact_user_rate_within_segment']:.1%}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
chart_path = os.path.join(ADDON_DIR, "04_strict_contact_user_rate.png")
plt.savefig(chart_path, dpi=160)
plt.show()
print("[SAVE]", chart_path)

# ---------- Chart 5: Positive mix strict vs other_interaction ----------
mix_df = plot_df[[
    "behavior_segment",
    "strict_share_of_segment_positive",
    "other_share_of_segment_positive"
]].copy()

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(mix_df))
width = 0.36

ax.bar(
    x - width / 2,
    mix_df["strict_share_of_segment_positive"],
    width,
    label="Strict contact share inside positive"
)

ax.bar(
    x + width / 2,
    mix_df["other_share_of_segment_positive"],
    width,
    label="Other interaction share inside positive"
)

ax.set_title("Inside positive events: strict contact vs other_interaction")
ax.set_xlabel("Behavior segment")
ax.set_ylabel("Share inside positive events")
ax.set_xticks(x)
ax.set_xticklabels(mix_df["behavior_segment"].astype(str), rotation=25, ha="right")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.legend()

plt.tight_layout()
chart_path = os.path.join(ADDON_DIR, "05_strict_vs_other_inside_positive.png")
plt.savefig(chart_path, dpi=160)
plt.show()
print("[SAVE]", chart_path)

# ============================================================
# 7. PRINT BUSINESS READING GUIDE
# ============================================================

print("\nDONE.")
print("Output folder:", ADDON_DIR)

print("""
CÁCH ĐỌC INSIGHT:

1) Mở bảng:
   09_segment_strict_contact_productivity.csv

2) Cột quan trọng:
   - user_share:
     Segment chiếm bao nhiêu % users.

   - strict_contact_share:
     Segment đóng góp bao nhiêu % contact thật
     = view_phone + contact_chat + contact_zalo + contact_sms.

   - strict_contact_contribution_index:
     = strict_contact_share / user_share.

     Nếu > 1:
       segment tạo contact nhiều hơn tỷ trọng user của nó.
       Ví dụ user_share = 8%, strict_contact_share = 20%
       => index = 2.5x, nhóm này over-contribute contact.

     Nếu < 1:
       segment đông user nhưng tạo contact ít hơn kỳ vọng.

3) Cách kể business:
   - Nếu focused_demand chiếm 80% users nhưng chỉ tạo 60% strict contacts:
     focused_demand đông nhưng không phải nguồn contact duy nhất.
   - Nếu multi_market_broker_like chiếm 8% users nhưng tạo 20% strict contacts:
     broker-like users đang over-contribute lead/contact.
   - Nếu broad_searcher có strict_contact_contribution_index thấp:
     nhóm này tìm rộng nhưng chưa tạo nhiều contact thật.
   - Nếu rental_broker_like nhỏ user_share nhưng contact index cao:
     thị trường thuê có nhóm user nhỏ nhưng hoạt động liên hệ rất mạnh.

4) Biểu đồ nên đưa vào slide:
   - 01_user_share_vs_strict_contact_share.png
   - 02_strict_contact_contribution_index.png
   - 03_strict_contacts_per_user.png
""")

print("Saved files:")
for f in sorted(os.listdir(ADDON_DIR)):
    print("-", f)

DB_PATH: /kaggle/working/eda_category_adtype_broker_like/category_adtype_broker_like.duckdb
PROFILE_PARQUET: /kaggle/working/eda_category_adtype_broker_like/user_positive_category_ad_profile_lite.parquet
ADDON_DIR: /kaggle/working/eda_category_adtype_broker_like/segment_strict_contact_addon


FileNotFoundError: Không thấy table user_positive_category_ad_profile trong DB và cũng không thấy user_positive_category_ad_profile_lite.parquet. Bạn cần chạy xong code LITE trước.